<a href="https://colab.research.google.com/github/krishnashashanth-sks/aiml-workspace/blob/main/NeRF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Understand NeRF Core Concepts (Advanced)

### Subtask:
Provide a comprehensive overview of Neural Radiance Fields (NeRF), emphasizing advanced concepts such as positional encoding's role in high-frequency detail, the multi-layer perceptron (MLP) architecture with skip connections, the importance of view-dependent color prediction, hierarchical volume sampling (coarse and fine networks), and the detailed volume rendering equation for transparency accumulation.


### 1. Core Idea of NeRF

Neural Radiance Fields (NeRF) represent a 3D scene as a continuous volumetric function. This function takes a 5D input (3D spatial coordinates (x, y, z) and 2D viewing directions (θ, φ)) and outputs the volume density (σ) and view-dependent emitted color (RGB) at that point in space. Essentially, NeRF learns to predict what color and how opaque any given point in a 3D scene is, when viewed from a specific direction.

### 2. Positional Encoding

Positional encoding is a critical component in NeRF for capturing high-frequency details. Standard MLPs struggle to learn high-frequency functions (functions that change rapidly), leading to blurry or oversmoothed renderings. To overcome this, positional encoding transforms the raw input coordinates (x, y, z) and view directions (θ, φ) into a higher-dimensional space using sinusoidal functions. Specifically, each scalar coordinate $p$ (e.g., $x, y, z, \theta, \phi$) is mapped to a vector:

$ \gamma(p) = (\sin(2^0 \pi p), \cos(2^0 \pi p), \sin(2^1 \pi p), \cos(2^1 \pi p), \dots, \sin(2^{L-1} \pi p), \cos(2^{L-1} \pi p)) $

where $L$ is the maximum frequency. This transformation allows the MLP to operate on inputs that are oscillating at different frequencies, making it easier for the network to learn complex, high-frequency variations in color and density, which are essential for rendering sharp edges, textures, and fine geometric details.

### 3. Multi-Layer Perceptron (MLP) Architecture with Skip Connections

The NeRF model employs a single, large MLP to approximate the continuous volumetric function. The architecture typically consists of 8 fully-connected layers, each with 256 ReLU activated units, followed by an output layer. A key feature of this MLP is the inclusion of **skip connections**. After an initial set of layers (e.g., the first 4-5 layers), the original positional encoded 3D input coordinates are concatenated with the features from an intermediate layer and fed into subsequent layers. This mechanism allows the network to bypass some layers, directly propagating the original input features to deeper parts of the network. This is crucial for two main reasons:

1.  **Mitigating Vanishing Gradients**: It helps combat the vanishing gradient problem in deep networks, ensuring that gradients can flow effectively during training.
2.  **Learning Multi-scale Features**: It enables the network to simultaneously learn both low-level features (from earlier layers directly connected to the input) and high-level, abstract features (from deeper layers), improving the network's ability to represent details at various scales.

### 4. View-Dependent Color Prediction

View-dependent color prediction is vital for generating photorealistic images, especially for capturing effects like reflections and specular highlights that change with the viewing angle. The NeRF MLP processes its inputs in two stages:

1.  **Density and Feature Vector Prediction**: The MLP first takes the positional encoded 3D coordinates (x, y, z) as input and processes them through several layers (typically 8 layers with 256 units). The output of this initial stage is the volume density (σ) and an intermediate feature vector. The density (σ) is view-independent, meaning it describes how much 'stuff' is at that location regardless of how you look at it.
2.  **Color Prediction**: The intermediate feature vector is then concatenated with the positional encoded 2D viewing direction (θ, φ). This combined input is passed through a few additional layers (e.g., 1 more layer with 128 units) to predict the RGB color. By conditioning the color prediction on the viewing direction, NeRF can accurately model effects like specular reflections, which appear differently depending on the viewer's perspective, leading to much more realistic renderings than view-independent models.

### 5. Hierarchical Volume Sampling (Coarse and Fine Networks)

To efficiently query the NeRF MLP and achieve high-quality renderings, NeRF employs a hierarchical volume sampling strategy involving two networks: a coarse network and a fine network. This addresses the inefficiency of uniform sampling, which would waste computation on empty space or occluded regions.

**a. Coarse Network Sampling**:

*   **Process**: For each ray cast through the scene, the coarse network uniformly samples a fixed number of points (e.g., 64 samples) along the ray's path. For each sampled point, the coarse network's MLP predicts its volume density (σ) and color (RGB).
*   **Purpose**: This initial sampling provides a rough estimate of the scene's geometry and appearance along the ray. It identifies regions where content is likely to exist.

**b. Fine Network Sampling**:

*   **Process**: Based on the density (σ) predictions from the coarse network, NeRF constructs an inverse transform sampling distribution. This distribution is highly concentrated in regions where the coarse network predicted high density (i.e., where surfaces or opaque objects are likely to be). A second set of points (e.g., 128 samples) are then adaptively sampled along the ray, biased towards these higher density regions. These new, adaptively sampled points are then passed to the separate 'fine' network, which is also an MLP (often with the same architecture as the coarse network but separately trained).
*   **Purpose**: The fine network refines the details in crucial areas, focusing computational resources where they matter most. This adaptive sampling significantly improves rendering quality and efficiency by concentrating samples in relevant parts of the scene, allowing the fine network to learn more precise representations of geometry and texture.

### 6. Volume Rendering Equation for Transparency Accumulation

To convert the predicted densities and colors from the NeRF MLP into a 2D image, NeRF uses classical volume rendering techniques. For each ray cast from the camera through a pixel, the final pixel color $C(\mathbf{r})$ is accumulated using the following integral:

$ C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \sigma(\mathbf{r}(t)) \mathbf{c}(\mathbf{r}(t), \mathbf{d}) dt $

where:

*   $ \mathbf{r} $ is the camera ray, parameterized by $ \mathbf{r}(t) = \mathbf{o} + t\mathbf{d} $, where $ \mathbf{o} $ is the ray origin and $ \mathbf{d} $ is the ray direction.
*   $ t_n $ and $ t_f $ are the near and far bounds of the ray.
*   $ \sigma(\mathbf{r}(t)) $ is the volume density predicted by the NeRF MLP at point $ \mathbf{r}(t) $. Higher density means more opacity.
*   $ \mathbf{c}(\mathbf{r}(t), \mathbf{d}) $ is the view-dependent RGB color predicted by the NeRF MLP at point $ \mathbf{r}(t) $ when viewed from direction $ \mathbf{d} $.
*   $ T(t) $ is the accumulated transmittance along the ray from $ t_n $ to $ t $, representing the probability that a ray travels from $ t_n $ to $ t $ without being absorbed. It is given by:

    $ T(t) = \exp\left(-\int_{t_n}^t \sigma(\mathbf{r}(s)) ds\right) $

In practice, this continuous integral is approximated using numerical quadrature, specifically by discretizing the ray into $N$ samples and using alpha compositing:

$ \hat{C}(\mathbf{r}) = \sum_{i=1}^N T_i (1 - \exp(-\sigma_i \delta_i)) \mathbf{c}_i $

where:

*   $ \delta_i $ is the distance between adjacent samples.
*   $ \sigma_i $ and $ \mathbf{c}_i $ are the density and color predicted by the MLP for sample $i$.
*   $ T_i $ is the discrete accumulated transmittance up to sample $i$, approximated as $ T_i = \exp(-\sum_{j=1}^{i-1} \sigma_j \delta_j) $ or $ T_i = \prod_{j=1}^{i-1} (1 - \alpha_j) $ where $ \alpha_j = 1 - \exp(-\sigma_j \delta_j) $ is the alpha (opacity) of sample $j$.

This rendering equation allows NeRF to blend colors and account for transparency as the ray passes through the volumetric scene representation, ultimately producing the final pixel color seen by the camera.

## Set Up Environment

### Subtask:
Install necessary Python libraries including `torch`, `numpy`, `matplotlib`, `tqdm`, and potentially `scikit-image` or `opencv-python` for data loading and preprocessing, ensuring a robust environment for advanced NeRF development.


**Reasoning**:
The first step to setting up the environment is to install the required libraries: `torch`, `numpy`, `matplotlib`, `tqdm`, and `scikit-image` using pip.



In [ ]:
!pip install torch numpy matplotlib tqdm scikit-image
print("Required libraries installed.")

Required libraries installed.


## Define Positional Encoding

### Subtask:
Implement the `PositionalEncoder` module using `torch.nn.Module`. This module will transform input 3D coordinates (x,y,z) and view directions (theta, phi) into high-dimensional feature vectors using sine and cosine functions at different frequencies, which is crucial for NeRF's ability to capture fine geometric and appearance details.


**Reasoning**:
The subtask requires implementing the `PositionalEncoder` module. I will define the `PositionalEncoder` class inheriting from `nn.Module`, implementing the `__init__` method to set up frequency bands and the `forward` method to apply sinusoidal functions and concatenate the results, as specified in the instructions.



In [ ]:
import torch
import torch.nn as nn
class PositionalEncoder(nn.Module):
  def __init__(self,N_freqs,log_space=True):
    super().__init__()
    self.N_freqs=N_freqs
    self.log_space=log_space
    if log_space:
      self.freq_bands=2.**torch.linspace(0.,N_freqs-1,N_freqs)*torch.pi
    else:
      self.freq_bands=torch.linspace(1.,2.**(N_freqs-1),N_freqs)*torch.pi
  def forward(self,x):
    x_expanded=x.unsqueeze(-1)
    x_modulated=x_expanded*self.freq_bands.to(x.device)
    sin_encoded=torch.sin(x_modulated)
    cos_encoded=torch.cos(x_modulated)
    sin_encoded=sin_encoded.flatten(start_dim=-2)
    cos_encoded=cos_encoded.flatten(start_dim=-2)
    return torch.cat([x,sin_encoded,cos_encoded],dim=-1)

In [ ]:
print("Defined the 'PositionalEncoder' class.")

# Example usage:
# Test with a 3D coordinate input
encoder = PositionalEncoder(N_freqs=10)
input_coords = torch.randn(1, 3) # Example: batch_size=1, 3D coordinates
encoded_coords = encoder(input_coords)
print(f"Original coordinates shape: {input_coords.shape}")
print(f"Encoded coordinates shape: {encoded_coords.shape}")

# Test with a 2D view direction input
encoder_view = PositionalEncoder(N_freqs=4)
input_viewdir = torch.randn(1, 2) # Example: batch_size=1, 2D view directions
encoded_viewdir = encoder_view(input_viewdir)
print(f"Original view direction shape: {input_viewdir.shape}")
print(f"Encoded view direction shape: {encoded_viewdir.shape}")

Defined the 'PositionalEncoder' class.
Original coordinates shape: torch.Size([1, 3])
Encoded coordinates shape: torch.Size([1, 63])
Original view direction shape: torch.Size([1, 2])
Encoded view direction shape: torch.Size([1, 18])


# Task
## Define NeRF MLP Architecture (Advanced)

### Subtask:
Create the core `NeRF` neural network architecture using `torch.nn.Module`. This MLP should be deep (e.g., 8 layers) with skip connections. It will take position embeddings as input to output density and intermediate features, which are then concatenated with view direction embeddings to predict view-dependent RGB color.

### Reasoning:
The task requires implementing the main NeRF Multi-Layer Perceptron (MLP) architecture. This involves defining a `torch.nn.Module` class that processes positional encoded 3D coordinates and 2D viewing directions to predict color and density. The architecture will include multiple linear layers, ReLU activations, and a skip connection as described in the NeRF paper. We will calculate the input dimensions for the MLP based on the previously defined `PositionalEncoder` and typical `N_freqs` values.

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assuming PositionalEncoder class and 'device' variable are defined in previous cells.

class NeRFMLP(nn.Module):
    """
    Implements the core NeRF MLP architecture.

    Args:
        pos_input_dim (int): Dimensionality of the positional encoded 3D coordinates.
        dir_input_dim (int): Dimensionality of the positional encoded 2D view directions.
        D (int): Number of layers in the main position branch.
        W (int): Width of each layer (number of neurons).
        skips (list): List of layer indices where a skip connection is applied.
    """
    def __init__(self, pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]):
        super(NeRFMLP, self).__init__()
        self.D = D # Number of layers for the position branch
        self.W = W # Width of each layer
        self.skips = skips # Layers at which to apply skip connection for position branch

        # Position encoding branch (for density and intermediate feature vector)
        self.pts_linears = nn.ModuleList()
        for i in range(D):
            if i == 0:
                layer_in_dim = pos_input_dim
            elif i in self.skips:
                # For skip connections, concatenate original input with layer output
                layer_in_dim = W + pos_input_dim
            else:
                layer_in_dim = W
            self.pts_linears.append(nn.Linear(layer_in_dim, W))

        # Output layers from the position branch
        self.feature_linear = nn.Linear(W, W) # Output intermediate feature vector
        self.alpha_linear = nn.Linear(W, 1)   # Output density (sigma)

        # View direction encoding branch (for RGB color)
        # Takes the intermediate feature vector and the view direction embedding as input
        self.views_linear = nn.Linear(W + dir_input_dim, W // 2)
        self.rgb_linear = nn.Linear(W // 2, 3) # Output RGB color

    def forward(self, x, d):
        """
        Forward pass through the NeRF MLP.

        Args:
            x (torch.Tensor): Positional encoded 3D coordinates (batch_size, pos_input_dim).
            d (torch.Tensor): Positional encoded 2D view directions (batch_size, dir_input_dim).

        Returns:
            tuple: A tuple containing:
                - rgb (torch.Tensor): Predicted RGB color (batch_size, 3), values between 0 and 1.
                - alpha (torch.Tensor): Predicted density (batch_size, 1), non-negative.
        """
        h = x # Initial input for the position branch
        for i, l in enumerate(self.pts_linears):
            h = l(h) # Apply linear layer
            h = F.relu(h) # Apply ReLU activation
            if i in self.skips:
                h = torch.cat([x, h], -1) # Apply skip connection by concatenating original input

        # Predict density (sigma) and an intermediate feature vector from the position branch's output
        alpha = self.alpha_linear(h)
        feature = self.feature_linear(h)

        # Concatenate the intermediate feature vector with the view direction embedding
        # and pass through the view-dependent color branch
        h_rgb = torch.cat([feature, d], -1)
        h_rgb = self.views_linear(h_rgb)
        h_rgb = F.relu(h_rgb)
        rgb = self.rgb_linear(h_rgb)

        # Apply sigmoid to RGB output to ensure values are between 0 and 1
        rgb = torch.sigmoid(rgb)
        # Apply ReLU to alpha output to ensure non-negative density
        alpha = F.relu(alpha)

        return rgb, alpha

# --- Calculate input dimensions based on previously defined PositionalEncoder ---
# These are typical N_freqs values for position and view direction in NeRF
N_freqs_pos = 10
N_freqs_dir = 4

# Instantiate PositionalEncoders to calculate output dimensions
# Ensure 'device' is available from previous cells (e.g., 'cpu' or 'cuda')
encoder_pos_test = PositionalEncoder(N_freqs=N_freqs_pos).to(device)
encoder_dir_test = PositionalEncoder(N_freqs=N_freqs_dir).to(device)

# Use dummy inputs to get the output shape of the PositionalEncoder
dummy_pos_input = torch.randn(1, 3).to(device) # 3D coordinates (x,y,z)
encoded_pos_output = encoder_pos_test(dummy_pos_input)
pos_input_dim = encoded_pos_output.shape[-1] # Expected: 3 + 2*3*N_freqs_pos = 63

dummy_dir_input = torch.randn(1, 2).to(device) # 2D view directions (theta, phi)
encoded_dir_output = encoder_dir_test(dummy_dir_input)
dir_input_dim = encoded_dir_output.shape[-1] # Expected: 2 + 2*2*N_freqs_dir = 18

print(f"Calculated pos_input_dim from PositionalEncoder with N_freqs={N_freqs_pos}: {pos_input_dim}")
print(f"Calculated dir_input_dim from PositionalEncoder with N_freqs={N_freqs_dir}: {dir_input_dim}")

# --- Instantiate the NeRF MLP model ---
nerf_model = NeRFMLP(pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]).to(device)

print("\nDefined NeRFMLP architecture and instantiated it.")
print(f"NeRF model architecture:\n{nerf_model}")

# --- Test forward pass with dummy data to verify output shapes ---
batch_size_test = 2 # Example batch size
test_pos_encoded = torch.randn(batch_size_test, pos_input_dim).to(device)
test_dir_encoded = torch.randn(batch_size_test, dir_input_dim).to(device)

rgb_out, alpha_out = nerf_model(test_pos_encoded, test_dir_encoded)

print(f"\nTest forward pass output shapes:")
print(f"RGB output shape: {rgb_out.shape}")   # Expected: (batch_size_test, 3)
print(f"Alpha output shape: {alpha_out.shape}") # Expected: (batch_size_test, 1)

assert rgb_out.shape == (batch_size_test, 3)
assert alpha_out.shape == (batch_size_test, 1)

print("NeRFMLP forward pass successful with correct output shapes.")
```

## Define NeRF MLP Architecture (Advanced)

### Subtask:
Create the core `NeRF` neural network architecture using `torch.nn.Module`. This MLP should be deep (e.g., 8 layers) with skip connections. It will take position embeddings as input to output density and intermediate features, which are then concatenated with view direction embeddings to predict view-dependent RGB color.


**Reasoning**:
I need to define the `NeRFMLP` class as instructed, including its `__init__` and `forward` methods, and then calculate its input dimensions using the previously defined `PositionalEncoder`. Finally, I will instantiate the `NeRFMLP` and perform a test forward pass.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NeRFMLP(nn.Module):
  def __init__(self,pos_input_dim,dir_input_dim,D=8,W=256,skips=[4]):
    super().__init__()
    self.D=D
    self.W=W
    self.skips=skips

    # The first layer always takes pos_input_dim
    self.pts_linears = nn.ModuleList([nn.Linear(pos_input_dim, W)])

    # Subsequent layers consider skip connections
    for i in range(1, D):
        layer_in_dim = W
        if i in self.skips:
            layer_in_dim += pos_input_dim # If skip, it will concatenate original input
        self.pts_linears.append(nn.Linear(layer_in_dim, W))

    self.alpha_linear=nn.Linear(W,1)
    self.feature_linear=nn.Linear(W,W)

    # View-dependent branch
    self.views_linears=nn.ModuleList([nn.Linear(W+dir_input_dim,W//2)])
    self.views_linears.extend([nn.Linear(W//2,W//2) for _ in range(2)])
    self.rgb_linear=nn.Linear(W//2,3)

  def forward(self,x,d):
    # x is the original positional encoded 3D coordinates
    # d is the original positional encoded 2D view directions
    h = x # 'h' will be the feature vector passed through the position branch

    for i,l in enumerate(self.pts_linears):
      # If this is a skip layer, concatenate the original input 'x' with 'h' before passing to the linear layer
      if i in self.skips:
        h = torch.cat([x, h], -1)

      h=l(h)
      h=F.relu(h)

    alpha=self.alpha_linear(h) # Output density
    feature=self.feature_linear(h) # Intermediate feature vector

    # Concatenate intermediate feature with view direction encoding for RGB prediction
    h_views=torch.cat([feature,d],-1)
    for i,l in enumerate(self.views_linears):
      h_views=l(h_views)
      h_views=F.relu(h_views)
    rgb=self.rgb_linear(h_views)

    rgb=torch.sigmoid(rgb) # Ensure RGB values are between 0 and 1
    alpha=F.relu(alpha)   # Ensure density is non-negative

    return rgb,alpha

In [ ]:
# 4. Calculate required input dimensions
encoder_pos = PositionalEncoder(N_freqs=10)
encoder_dir = PositionalEncoder(N_freqs=4)
device=torch.device("cuda" if torch.cuda.is_available()else 'cpu')
dummy_coords = torch.randn(1, 3)
dummy_viewdir = torch.randn(1, 2)

pos_input_dim = encoder_pos(dummy_coords).shape[-1]
dir_input_dim = encoder_dir(dummy_viewdir).shape[-1]

print(f"Calculated pos_input_dim: {pos_input_dim}")
print(f"Calculated dir_input_dim: {dir_input_dim}")

# 5. Instantiate the NeRFMLP model
nerf_model = NeRFMLP(pos_input_dim, dir_input_dim).to(device)
print("Instantiated NeRFMLP model.")
print(f"Model architecture:\n{nerf_model}")

# 6. Perform a test forward pass
batch_size_test = 64 # Using a common batch size for testing
dummy_encoded_pos = torch.randn(batch_size_test, pos_input_dim).to(device)
dummy_encoded_dir = torch.randn(batch_size_test, dir_input_dim).to(device)

output_rgb, output_alpha = nerf_model(dummy_encoded_pos, dummy_encoded_dir)

print(f"Output RGB shape: {output_rgb.shape}")
print(f"Output Alpha shape: {output_alpha.shape}")

# Verify shapes
assert output_rgb.shape == (batch_size_test, 3), f"Expected RGB shape ({batch_size_test}, 3), but got {output_rgb.shape}"
assert output_alpha.shape == (batch_size_test, 1), f"Expected Alpha shape ({batch_size_test}, 1), but got {output_alpha.shape}"
print("Test forward pass successful: output shapes are correct.")


Calculated pos_input_dim: 63
Calculated dir_input_dim: 18
Instantiated NeRFMLP model.
Model architecture:
NeRFMLP(
  (pts_linears): ModuleList(
    (0): Linear(in_features=63, out_features=256, bias=True)
    (1-3): 3 x Linear(in_features=256, out_features=256, bias=True)
    (4): Linear(in_features=319, out_features=256, bias=True)
    (5-7): 3 x Linear(in_features=256, out_features=256, bias=True)
  )
  (alpha_linear): Linear(in_features=256, out_features=1, bias=True)
  (feature_linear): Linear(in_features=256, out_features=256, bias=True)
  (views_linears): ModuleList(
    (0): Linear(in_features=274, out_features=128, bias=True)
    (1-2): 2 x Linear(in_features=128, out_features=128, bias=True)
  )
  (rgb_linear): Linear(in_features=128, out_features=3, bias=True)
)
Output RGB shape: torch.Size([64, 3])
Output Alpha shape: torch.Size([64, 1])
Test forward pass successful: output shapes are correct.


# Task
## Implement Hierarchical Ray Sampling: `get_rays`

### Subtask:
Develop the `get_rays` function. This function will generate ray origins and directions for each pixel in an image, given camera parameters such as image height, image width, focal length, and the camera-to-world transformation matrix (`c2w`).

### Reasoning:
To implement hierarchical ray sampling, the first logical step is to accurately define how rays are generated from a given camera pose. The `get_rays` function will leverage camera intrinsics (focal length, image dimensions) and extrinsics (camera-to-world matrix) to calculate the origin and direction of each ray passing through the center of each pixel. This function will be crucial for both coarse and fine sampling stages.

```python
import torch

def get_rays(H, W, focal, c2w):
    """
    Generate ray origins and directions for all pixels in an image.

    Args:
        H (int): Image height.
        W (int): Image width.
        focal (float): Focal length of the camera.
        c2w (torch.Tensor): Camera-to-world transformation matrix (4x4).

    Returns:
        tuple: A tuple containing:
            - rays_o (torch.Tensor): Ray origins for all pixels (H*W, 3).
            - rays_d (torch.Tensor): Ray directions for all pixels (H*W, 3).
    """
    # Create a grid of (x, y) pixel coordinates
    # The pixel coordinates are normalized and shifted to have (0,0) at the center
    i, j = torch.meshgrid(torch.linspace(0, W - 1, W), torch.linspace(0, H - 1, H), indexing='ij')
    i = i.t() # Transpose to get (H, W)
    j = j.t() # Transpose to get (H, W)

    # Calculate normalized pixel coordinates relative to the camera center
    # This assumes the camera's principal point is at (W/2, H/2)
    dirs = torch.stack([(i - W * 0.5) / focal,
                        -(j - H * 0.5) / focal, # Y-axis inverted in image coordinates
                        -torch.ones_like(i)], -1) # Z-axis points into the scene

    # Transform ray directions from camera frame to world frame
    # c2w[:3, :3] is the rotation matrix, dirs.unsqueeze(-2) adds a dimension for broadcasting
    rays_d = torch.sum(dirs[..., None, :] * c2w[:3, :3], -1)

    # Ray origins are simply the camera position in world coordinates, repeated for each pixel
    # c2w[:3, 3] is the translation vector (camera origin in world frame)
    rays_o = c2w[:3, -1].expand(rays_d.shape)

    return rays_o, rays_d

print("Defined the 'get_rays' function for generating ray origins and directions.")

# Example usage (dummy values for testing):
H_test, W_test = 100, 100
focal_test = 120.0
# Example camera-to-world matrix (identity for simplicity, meaning camera is at origin, looking down -Z)
# In a real scenario, this would come from a dataset or pose estimation.
c2w_test = torch.eye(4)
c2w_test[:3, 3] = torch.tensor([0.0, 0.0, -2.0]) # Example: camera at (0,0,-2), looking towards origin
c2w_test = c2w_test.to(device) # Ensure tensor is on the correct device

test_rays_o, test_rays_d = get_rays(H_test, W_test, focal_test, c2w_test)

print(f"Generated test ray origins shape: {test_rays_o.shape}") # Expected: (H*W, 3)
print(f"Generated test ray directions shape: {test_rays_d.shape}") # Expected: (H*W, 3)

assert test_rays_o.shape == (H_test * W_test, 3)
assert test_rays_d.shape == (H_test * W_test, 3)
print("Test `get_rays` function successful: output shapes are correct.")
```

## Implement Hierarchical Ray Sampling: get_rays

### Subtask:
Develop the `get_rays` function, which generates ray origins and directions for each pixel in an image using camera parameters.


**Reasoning**:
The subtask requires defining the `get_rays` function and demonstrating its usage with dummy parameters. I will implement the function as specified, create the dummy inputs, call the function, print the shapes of the outputs, and add assertions to verify the output shapes.



In [ ]:
import torch
def get_rays(H,W,focal,c2w):
  """
    Generates ray origins and directions for each pixel in an image.

    Args:
        H (int): Image height.
        W (int): Image width.
        focal (float): Focal length of the camera.
        c2w (torch.Tensor): Camera-to-world transformation matrix (4x4).

    Returns:
        tuple: A tuple containing:
            - rays_o (torch.Tensor): Ray origins in world coordinates (H*W, 3).
            - rays_d (torch.Tensor): Ray directions in world coordinates (H*W, 3).
  """
  i,j=torch.meshgrid(
      torch.linspace(0,W-1,W,device=c2w.device),
      torch.linspace(0,H-1,H,device=c2w.device),
      indexing='xy'
  )
  dirs=torch.stack([
      (i-W*0.5)/focal,
      -(j-H*0.5)/focal,
      -torch.ones_like(i)
  ],-1)
  rays_d=torch.sum(dirs[...,None,:]*c2w[:3,:3],-1)
  rays_o=c2w[:3,3].expand(rays_d.shape)
  rays_o=rays_o.reshape(-1,3)
  rays_d=rays_d.reshape(-1,3)
  return rays_o,rays_d

In [ ]:
# 2. Create dummy values
H_dummy = 100
W_dummy = 100
focal_dummy = 120.0

# Sample c2w matrix (identity with a translation)
# Ensure it's on the correct device (e.g., 'cpu' or 'cuda' as defined previously)
if 'device' not in globals(): # Fallback if 'device' variable is not defined in previous cells
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
c2w_dummy = torch.eye(4, device=device)
c2w_dummy[:3, 3] = torch.tensor([1.0, 2.0, 3.0], device=device) # Example translation

print(f"Dummy H: {H_dummy}, W: {W_dummy}, Focal: {focal_dummy}")
print(f"Dummy c2w matrix:\n{c2w_dummy}")

# 3. Call the get_rays function
rays_o_dummy, rays_d_dummy = get_rays(H_dummy, W_dummy, focal_dummy, c2w_dummy)

# 4. Print the shapes of the returned ray origins and ray directions
print(f"\nShape of generated ray origins: {rays_o_dummy.shape}")
print(f"Shape of generated ray directions: {rays_d_dummy.shape}")

# 5. Add assertions to verify that the shapes are (H*W, 3)
expected_shape = (H_dummy * W_dummy, 3)
assert rays_o_dummy.shape == expected_shape, f"Ray origins shape mismatch: Expected {expected_shape}, got {rays_o_dummy.shape}"
assert rays_d_dummy.shape == expected_shape, f"Ray directions shape mismatch: Expected {expected_shape}, got {rays_d_dummy.shape}"

print("Assertions passed: Ray origins and directions have the expected shapes.")

Dummy H: 100, W: 100, Focal: 120.0
Dummy c2w matrix:
tensor([[1., 0., 0., 1.],
        [0., 1., 0., 2.],
        [0., 0., 1., 3.],
        [0., 0., 0., 1.]], device='cuda:0')

Shape of generated ray origins: torch.Size([10000, 3])
Shape of generated ray directions: torch.Size([10000, 3])
Assertions passed: Ray origins and directions have the expected shapes.


## Implement Hierarchical Ray Sampling

### Subtask:
Develop the `sample_points_on_rays` function, which implements coarse and fine sampling along rays.


**Reasoning**:
The subtask requires defining the `sample_points_on_rays` function. I will implement the initial part of the function, which handles coarse sampling and the calculation of 3D points and depths (`z_vals_coarse`), as well as initializing the fine sampling variables. This step covers instructions 1 and 2.



# Task
Okay, I will complete the `sample_points_on_rays` function to include hierarchical ray sampling (fine sampling) and then test it thoroughly.

First, I need to remove the partially implemented `sample_points_on_ways` function from the temporary code cell `FIGi5Hxz__kgk`.
Then, I will add a new code cell and execute the complete implementation of `sample_points_on_rays` from the detailed task description in the notebook, which includes both coarse and fine sampling logic along with comprehensive testing.

This will directly address the plan items:
*   **Implement Hierarchical Ray Sampling (Fine Sampling)**
*   **Test Hierarchical Ray Sampling**

```python
# Delete the incomplete code cell: FIGi5Hxz__kgk
```
I will now add the complete `sample_points_on_rays` function and its tests in a new code cell.

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure 'device' is defined for tensor placement
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-define PositionalEncoder if it was not in the global scope previously
class PositionalEncoder(nn.Module):
    def __init__(self, N_freqs, log_space=True):
        super().__init__()
        self.N_freqs = N_freqs
        self.log_space = log_space
        if log_space:
            self.freq_bands = 2.**torch.linspace(0., N_freqs - 1, N_freqs) * torch.pi
        else:
            self.freq_bands = torch.linspace(1., 2.**(N_freqs - 1), N_freqs) * torch.pi

    def forward(self, x):
        x_expanded = x.unsqueeze(-1)
        x_modulated = x_expanded * self.freq_bands.to(x.device)
        sin_encoded = torch.sin(x_modulated)
        cos_encoded = torch.cos(x_modulated)
        sin_encoded = sin_encoded.flatten(start_dim=-2)
        cos_encoded = cos_encoded.flatten(start_dim=-2)
        return torch.cat([x, sin_encoded, cos_encoded], dim=-1)

# Re-define NeRFMLP if it was not in the global scope previously
class NeRFMLP(nn.Module):
    def __init__(self, pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]):
        super().__init__()
        self.D = D
        self.W = W
        self.skips = skips

        # This definition for pts_linears is more robust to D=0 cases,
        # but for D=8, the original explicit loop is fine.
        # However, to avoid potential issues with list comprehension and `self.skips` indexing
        # when D-1 < len(self.skips), let's keep it explicit for clarity.
        self.pts_linears = nn.ModuleList()
        for i in range(D):
            if i == 0:
                layer_in_dim = pos_input_dim
            elif i in self.skips:
                layer_in_dim = W + pos_input_dim
            else:
                layer_in_dim = W
            self.pts_linears.append(nn.Linear(layer_in_dim, W))

        self.alpha_linear = nn.Linear(W, 1)
        self.feature_linear = nn.Linear(W, W)
        self.views_linears = nn.ModuleList([nn.Linear(W + dir_input_dim, W // 2)])
        self.views_linears.extend([nn.Linear(W // 2, W // 2) for _ in range(2)]) # Use _ for unused loop variable
        self.rgb_linear = nn.Linear(W // 2, 3)

    def forward(self, x, d):
        h = x # The typo 'z' was corrected to 'x'
        for i, l in enumerate(self.pts_linears):
            h = l(h)
            h = F.relu(h)
            if i in self.skips:
                h = torch.cat([x, h], -1)
        alpha = self.alpha_linear(h)
        feature = self.feature_linear(h)
        h_views = torch.cat([feature, d], -1)
        for i, l in enumerate(self.views_linears):
            h_views = l(h_views)
            h_views = F.relu(h_views)
        rgb = self.rgb_linear(h_views)
        rgb = torch.sigmoid(rgb)
        alpha = F.relu(alpha)
        return rgb, alpha


def sample_points_on_rays(rays_o, rays_d, near, far, N_samples, N_importance=0, perturb=True,
                          lindisp=False, coarse_model=None, positional_encoder_pos=None, positional_encoder_dir=None):
    """
    Samples points along rays for both coarse and fine networks.

    Args:
        rays_o (torch.Tensor): Ray origins (N_rays, 3).
        rays_d (torch.Tensor): Ray directions (N_rays, 3).
        near (float): Near bound for ray sampling.
        far (float): Far bound for ray sampling.
        N_samples (int): Number of points to sample uniformly for the coarse network.
        N_importance (int): Number of points to sample adaptively for the fine network.
        perturb (bool): Whether to add stochastic perturbation to samples.
        lindisp (bool): If true, sample in inverse depth (disparity) space.
        coarse_model (NeRFMLP): The coarse NeRF model for density prediction.
        positional_encoder_pos (PositionalEncoder): Positional encoder for 3D coordinates.
        positional_encoder_dir (PositionalEncoder): Positional encoder for 2D view directions.


    Returns:
        tuple: A tuple containing:
            - pts (torch.Tensor): Sampled 3D points (N_rays, N_samples + N_importance, 3).
            - z_vals (torch.Tensor): Depths of sampled points along rays (N_rays, N_samples + N_importance).
    """
    N_rays = rays_o.shape[0]

    # 1. Coarse Sampling (Uniform or Linear in Disparity)
    t_vals = torch.linspace(0., 1., N_samples, device=rays_o.device)
    if lindisp:
        # Sample linearly in disparity (inverse depth) space
        t_vals = 1. / (1. / near * (1. - t_vals) + 1. / far * t_vals)
    else:
        # Sample linearly in depth space
        t_vals = near * (1. - t_vals) + far * t_vals

    if perturb:
        # Add uniform noise to each sample along the ray
        # This prevents samples from always falling at the same relative positions
        mids = .5 * (t_vals[..., 1:] + t_vals[..., :-1])
        upper = torch.cat([mids, t_vals[..., -1:]], -1)
        lower = torch.cat([t_vals[..., :1], mids], -1)
        # Randomly sample between adjacent `t_vals`
        t_rand = torch.rand(N_samples, device=rays_o.device)
        t_vals = lower + (upper - lower) * t_rand

    # Expand t_vals to (N_rays, N_samples) and add ray origins and directions
    # pts = o + t * d
    pts_coarse = rays_o[..., None, :] + rays_d[..., None, :] * t_vals[..., :, None] # (N_rays, N_samples, 3)
    z_vals_coarse = t_vals.expand(N_rays, N_samples)

    if N_importance > 0 and coarse_model is not None and positional_encoder_pos is not None and positional_encoder_dir is not None:
        # 2. Fine Sampling (Adaptive based on coarse densities)
        with torch.no_grad():
            # Get densities from the coarse model
            # Reshape pts_coarse for MLP input: (N_rays * N_samples, 3)
            pts_flat = pts_coarse.reshape(-1, 3)
            # Encode points
            encoded_pts_flat = positional_encoder_pos(pts_flat)
            # Duplicate ray directions for each sampled point
            rays_d_expanded = rays_d[..., None, :].expand(-1, N_samples, -1).reshape(-1, 3)
            encoded_rays_d_expanded = positional_encoder_dir(rays_d_expanded) # Assuming positional_encoder_dir is available
            # Predict RGB and alpha (density)
            _, raw_alpha = coarse_model(encoded_pts_flat, encoded_rays_d_expanded) # (N_rays * N_samples, 1)

            # Convert raw alpha (density) to weights for sampling
            # sigma_i = raw_alpha
            # alpha_i = 1 - exp(-sigma_i * delta_i)
            # weights_i = T_i * alpha_i
            # T_i = exp(-sum(sigma_j * delta_j))
            # delta_i are distances between adjacent samples
            dists = torch.cat([z_vals_coarse[..., 1:] - z_vals_coarse[..., :-1],
                               torch.tensor([1e10], device=rays_o.device).expand(z_vals_coarse[..., :1].shape)], -1)
            dists = dists.reshape(N_rays * N_samples, 1) # (N_rays * N_samples, 1)

            # (N_rays * N_samples, 1)
            alpha = 1. - torch.exp(-raw_alpha * dists)
            # (N_rays, N_samples)
            alpha = alpha.reshape(N_rays, N_samples)
            # (N_rays, N_samples) - Calculate transmittance
            weights = alpha * torch.cumprod(torch.cat([torch.ones((N_rays, 1), device=rays_o.device), 1.-alpha + 1e-10], -1), -1)[:, :-1]
            weights = weights + 1e-5 # Add small value to prevent division by zero

            # Normalize weights to form a PDF
            pdf = weights / torch.sum(weights, -1, keepdim=True) # (N_rays, N_samples)
            cdf = torch.cumsum(pdf, -1) # (N_rays, N_samples)
            cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], -1) # (N_rays, N_samples + 1)

        # Inverse transform sampling
        u = torch.rand(N_rays, N_importance, device=rays_o.device)
        u = u.contiguous() # Ensure contiguous memory for searchsorted
        # Find indices of bins where u would fall
        # `right=True` means that a value equal to the rightmost value will be placed to its right
        inds = torch.searchsorted(cdf, u, right=True)

        # Get values of cdf and z_vals at these indices
        below = torch.max(torch.zeros_like(inds - 1), inds - 1)
        above = torch.min((cdf.shape[-1] - 1) * torch.ones_like(inds), inds)
        # Reshape to (N_rays, N_importance, 1) for gathering
        inds_g = torch.stack([below, above], -1) # (N_rays, N_importance, 2)

        matched_shape = [inds_g.shape[0], inds_g.shape[1], cdf.shape[-1]]
        # The gather operation needs the cdf and z_vals to have a batch dimension matching inds_g
        # Using a direct indexing approach for clarity, assuming inds_g contains valid indices
        # Simplified indexing if tensors are 2D and inds_g is (N_rays, N_importance, 2)
        # However, torch.gather needs the `index` tensor's dimensions to be broadcastable or match `input`
        # Let's use advanced indexing which is often clearer for this
        
        # Original logic for gathering
        cdf_g_0 = torch.gather(cdf, -1, inds_g[..., 0])
        cdf_g_1 = torch.gather(cdf, -1, inds_g[..., 1])
        z_vals_g_0 = torch.gather(z_vals_coarse, -1, inds_g[..., 0])
        z_vals_g_1 = torch.gather(z_vals_coarse, -1, inds_g[..., 1])

        # Calculate inverse transform sample points
        denom = cdf_g_1 - cdf_g_0
        denom = torch.where(denom < 1e-5, torch.ones_like(denom), denom) # Prevent division by zero
        t = (u - cdf_g_0) / denom
        z_vals_fine = z_vals_g_0 + t * (z_vals_g_1 - z_vals_g_0)


        # Combine coarse and fine samples and sort them
        z_vals_combined, _ = torch.sort(torch.cat([z_vals_coarse, z_vals_fine], -1), -1)

        # Compute 3D points for combined z_vals
        pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals_combined[..., :, None]
        return pts, z_vals_combined
    else:
        # Only return coarse samples if fine sampling is not requested or not possible
        return pts_coarse, z_vals_coarse


print("Defined the 'sample_points_on_rays' function for hierarchical sampling.")

# --- Test `sample_points_on_rays` function ---
# Assume 'device' is already defined ('cpu' or 'cuda')
# Assume 'get_rays' is already defined from previous steps

# Dummy camera parameters
H_test, W_test = 10, 10 # Small resolution for quick testing
focal_test = 50.0
c2w_test = torch.eye(4, device=device)
c2w_test[:3, 3] = torch.tensor([0.0, 0.0, -4.0], device=device) # Camera at (0,0,-4)

# Generate rays
test_rays_o, test_rays_d = get_rays(H_test, W_test, focal_test, c2w_test)
N_rays_test = test_rays_o.shape[0]

# Define near and far bounds
near_test, far_test = 2.0, 6.0

# Coarse sampling parameters
N_samples_coarse = 64
N_samples_fine = 128 # Set to 0 to test only coarse sampling

print(f"\nTesting `sample_points_on_rays` with N_rays={N_rays_test}, N_samples_coarse={N_samples_coarse}, N_samples_fine={N_samples_fine}...")

# Test without fine sampling first
pts_coarse_only, z_vals_coarse_only = sample_points_on_rays(
    test_rays_o, test_rays_d, near_test, far_test, N_samples=N_samples_coarse, N_importance=0, perturb=True, lindisp=False,
    coarse_model=None, positional_encoder_pos=None, positional_encoder_dir=None # Explicitly set to None for coarse-only
)

print(f"Shape of points from coarse sampling only: {pts_coarse_only.shape}")
print(f"Shape of depths from coarse sampling only: {z_vals_coarse_only.shape}")
assert pts_coarse_only.shape == (N_rays_test, N_samples_coarse, 3)
assert z_vals_coarse_only.shape == (N_rays_test, N_samples_coarse)
print("Coarse sampling only test successful.")

# Test with both coarse and fine sampling
# Need dummy NeRF model and positional encoders for fine sampling
N_freqs_pos = 10
N_freqs_dir = 4
pos_encoder = PositionalEncoder(N_freqs=N_freqs_pos).to(device)
dir_encoder = PositionalEncoder(N_freqs=N_freqs_dir).to(device) # Need for model initialization

# Calculate dimensions for NeRFMLP
pos_input_dim = pos_encoder(torch.randn(1, 3).to(device)).shape[-1]
dir_input_dim = dir_encoder(torch.randn(1, 3).to(device)).shape[-1] # View direction is 3D vector after normalization

coarse_nerf_model = NeRFMLP(pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]).to(device)

# Provide positional_encoder_dir to the function call
pts_combined, z_vals_combined = sample_points_on_rays(
    test_rays_o, test_rays_d, near_test, far_test, N_samples=N_samples_coarse,
    N_importance=N_samples_fine, perturb=True, lindisp=False,
    coarse_model=coarse_nerf_model, positional_encoder_pos=pos_encoder, positional_encoder_dir=dir_encoder
)

print(f"Shape of points from combined sampling: {pts_combined.shape}")
print(f"Shape of depths from combined sampling: {z_vals_combined.shape}")
assert pts_combined.shape == (N_rays_test, N_samples_coarse + N_samples_fine, 3)
assert z_vals_combined.shape == (N_rays_test, N_samples_coarse + N_samples_fine)
print("Combined (coarse + fine) sampling test successful.")
print("`sample_points_on_rays` function fully tested.")
```

## Implement Hierarchical Ray Sampling (Fine Sampling)

### Subtask:
Complete the `sample_points_on_rays` function by implementing the fine sampling logic, using the coarse model's density predictions to construct a PDF, performing inverse transform sampling, and combining the coarse and fine samples. Also, thoroughly test the function.


**Reasoning**:
I need to complete the `sample_points_on_rays` function by implementing the fine sampling logic, which involves generating weights from coarse densities, constructing a CDF, and performing inverse transform sampling. After implementing the full function, I will thoroughly test it as specified in the instructions, including both coarse-only and combined sampling scenarios. I will also ensure all necessary helper classes and functions are defined within the code block for execution.



# Task
Okay, I will re-run the testing for `NeRFMLP` and `get_rays` to confirm the fixes.

## Re-run testing for `NeRFMLP` and `get_rays`

### Subtask:
Execute the testing code associated with `NeRFMLP` in cell `a9b96bcb` and the testing code for `get_rays` in cell `06de272e` to confirm that the fixes have resolved the shape mismatch and typo errors, and that both functions produce the expected outputs.


**Reasoning**:
The user explicitly requested to re-run the testing code for `NeRFMLP` and `get_rays` to confirm previous fixes. I will combine the relevant class definitions (`PositionalEncoder`, `NeRFMLP`) and function definition (`get_rays`) along with the testing snippets from cells `a9b96bcb` and `06de272e` into a single code block. This ensures that all necessary components are available and the tests can be executed successfully in an isolated manner, verifying the correctness of the implemented functions.



In [ ]:
# --- Tests for NeRFMLP (from cell a9b96bcb) ---
print("\n--- Starting NeRFMLP tests ---")
# 4. Calculate required input dimensions
encoder_pos = PositionalEncoder(N_freqs=10).to(device) # Ensure encoder is on device
encoder_dir = PositionalEncoder(N_freqs=4).to(device) # Ensure encoder is on device

dummy_coords = torch.randn(1, 3).to(device) # Ensure input is on device
dummy_viewdir = torch.randn(1, 2).to(device) # Ensure input is on device

pos_input_dim = encoder_pos(dummy_coords).shape[-1]
dir_input_dim = encoder_dir(dummy_viewdir).shape[-1]

print(f"Calculated pos_input_dim: {pos_input_dim}")
print(f"Calculated dir_input_dim: {dir_input_dim}")

# 5. Instantiate the NeRFMLP model
nerf_model = NeRFMLP(pos_input_dim, dir_input_dim).to(device)
print("Instantiated NeRFMLP model.")
print(f"Model architecture:\n{nerf_model}")

# 6. Perform a test forward pass
batch_size_test = 64 # Using a common batch size for testing
dummy_encoded_pos = torch.randn(batch_size_test, pos_input_dim).to(device)
dummy_encoded_dir = torch.randn(batch_size_test, dir_input_dim).to(device)

output_rgb, output_alpha = nerf_model(dummy_encoded_pos, dummy_encoded_dir)

print(f"Output RGB shape: {output_rgb.shape}")
print(f"Output Alpha shape: {output_alpha.shape}")

# Verify shapes
assert output_rgb.shape == (batch_size_test, 3), f"Expected RGB shape ({batch_size_test}, 3), but got {output_rgb.shape}"
assert output_alpha.shape == (batch_size_test, 1), f"Expected Alpha shape ({batch_size_test}, 1), but got {output_alpha.shape}"
print("Test forward pass successful: output shapes are correct.")
print("--- NeRFMLP tests completed ---")


# --- Tests for get_rays (from cell 06de272e) ---
print("\n--- Starting get_rays function tests ---")
# 2. Create dummy values
H_dummy = 100
W_dummy = 100
focal_dummy = 120.0

# Sample c2w matrix (identity with a translation)
c2w_dummy = torch.eye(4, device=device)
c2w_dummy[:3, 3] = torch.tensor([1.0, 2.0, 3.0], device=device) # Example translation

print(f"Dummy H: {H_dummy}, W: {W_dummy}, Focal: {focal_dummy}")
print(f"Dummy c2w matrix:\n{c2w_dummy}")

# 3. Call the get_rays function
rays_o_dummy, rays_d_dummy = get_rays(H_dummy, W_dummy, focal_dummy, c2w_dummy)

# 4. Print the shapes of the returned ray origins and ray directions
print(f"\nShape of generated ray origins: {rays_o_dummy.shape}")
print(f"Shape of generated ray directions: {rays_d_dummy.shape}")

# 5. Add assertions to verify that the shapes are (H*W, 3)
expected_shape = (H_dummy * W_dummy, 3)
assert rays_o_dummy.shape == expected_shape, f"Ray origins shape mismatch: Expected {expected_shape}, got {rays_o_dummy.shape}"
assert rays_d_dummy.shape == expected_shape, f"Ray directions shape mismatch: Expected {expected_shape}, got {rays_d_dummy.shape}"

print("Assertions passed: Ray origins and directions have the expected shapes.")
print("-- get_rays function tests completed ---")

print("\n--- All specified tests completed ---")



--- Starting NeRFMLP tests ---
Calculated pos_input_dim: 63
Calculated dir_input_dim: 18
Instantiated NeRFMLP model.
Model architecture:
NeRFMLP(
  (pts_linears): ModuleList(
    (0): Linear(in_features=63, out_features=256, bias=True)
    (1-3): 3 x Linear(in_features=256, out_features=256, bias=True)
    (4): Linear(in_features=319, out_features=256, bias=True)
    (5-7): 3 x Linear(in_features=256, out_features=256, bias=True)
  )
  (alpha_linear): Linear(in_features=256, out_features=1, bias=True)
  (feature_linear): Linear(in_features=256, out_features=256, bias=True)
  (views_linears): ModuleList(
    (0): Linear(in_features=274, out_features=128, bias=True)
    (1-2): 2 x Linear(in_features=128, out_features=128, bias=True)
  )
  (rgb_linear): Linear(in_features=128, out_features=3, bias=True)
)
Output RGB shape: torch.Size([64, 3])
Output Alpha shape: torch.Size([64, 1])
Test forward pass successful: output shapes are correct.
--- NeRFMLP tests completed ---

--- Starting get_

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure 'device' is defined for tensor placement
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def sample_points_on_rays(rays_o, rays_d, near, far, N_samples, N_importance=0, perturb=True,
                          lindisp=False, coarse_model=None, positional_encoder_pos=None, positional_encoder_dir=None):
    """
    Samples points along rays for both coarse and fine networks.

    Args:
        rays_o (torch.Tensor): Ray origins (N_rays, 3).
        rays_d (torch.Tensor): Ray directions (N_rays, 3).
        near (float): Near bound for ray sampling.
        far (float): Far bound for ray sampling.
        N_samples (int): Number of points to sample uniformly for the coarse network.
        N_importance (int): Number of points to sample adaptively for the fine network.
        perturb (bool): Whether to add stochastic perturbation to samples.
        lindisp (bool): If true, sample in inverse depth (disparity) space.
        coarse_model (NeRFMLP): The coarse NeRF model for density prediction.
        positional_encoder_pos (PositionalEncoder): Positional encoder for 3D coordinates.
        positional_encoder_dir (PositionalEncoder): Positional encoder for 2D view directions.


    Returns:
        tuple: A tuple containing:
            - pts (torch.Tensor): Sampled 3D points (N_rays, N_samples + N_importance, 3).
            - z_vals (torch.Tensor): Depths of sampled points along rays (N_rays, N_samples + N_importance).
    """
    N_rays = rays_o.shape[0]

    # 1. Coarse Sampling (Uniform or Linear in Disparity)
    t_vals = torch.linspace(0., 1., N_samples, device=rays_o.device) # (N_samples,)
    if lindisp:
        # Sample linearly in disparity (inverse depth) space
        t_vals = 1. / (1. / near * (1. - t_vals) + 1. / far * t_vals)
    else:
        # Sample linearly in depth space
        t_vals = near * (1. - t_vals) + far * t_vals

    if perturb:
        # Add uniform noise to each sample along the ray
        # This prevents samples from always falling at the same relative positions
        mids = .5 * (t_vals[..., 1:] + t_vals[..., :-1])
        upper = torch.cat([mids, t_vals[..., -1:]], -1)
        lower = torch.cat([t_vals[..., :1], mids], -1)
        # Randomly sample between adjacent `t_vals`
        t_rand = torch.rand(N_samples, device=rays_o.device)
        t_vals = lower + (upper - lower) * t_rand

    # Expand t_vals to (N_rays, N_samples) and add ray origins and directions
    # pts = o + t * d
    pts_coarse = rays_o[..., None, :] + rays_d[..., None, :] * t_vals[..., :, None] # (N_rays, N_samples, 3)
    z_vals_coarse = t_vals.expand(N_rays, N_samples)

    if N_importance > 0 and coarse_model is not None and positional_encoder_pos is not None and positional_encoder_dir is not None:
        # 2. Fine Sampling (Adaptive based on coarse densities)
        with torch.no_grad():
            # Get densities from the coarse model
            # Reshape pts_coarse for MLP input: (N_rays * N_samples, 3)
            pts_flat = pts_coarse.reshape(-1, 3)
            # Encode points
            encoded_pts_flat = positional_encoder_pos(pts_flat)
            # Duplicate ray directions for each sampled point
            rays_d_expanded = rays_d[..., None, :].expand(-1, N_samples, -1).reshape(-1, 3)
            encoded_rays_d_expanded = positional_encoder_dir(rays_d_expanded)
            # Predict RGB and alpha (density)
            _, raw_alpha = coarse_model(encoded_pts_flat, encoded_rays_d_expanded) # (N_rays * N_samples, 1)

            # Convert raw alpha (density) to weights for sampling
            # sigma_i = raw_alpha
            # alpha_i = 1 - exp(-sigma_i * delta_i)
            # weights_i = T_i * alpha_i
            # T_i = exp(-sum(sigma_j * delta_j))
            # delta_i are distances between adjacent samples
            dists = torch.cat([z_vals_coarse[..., 1:] - z_vals_coarse[..., :-1],
                               torch.tensor([1e10], device=rays_o.device).expand(z_vals_coarse[..., :1].shape)], -1)
            # dists is (N_rays, N_samples)
            # raw_alpha is (N_rays * N_samples, 1)
            # Need to reshape raw_alpha to (N_rays, N_samples)
            raw_alpha_reshaped = raw_alpha.reshape(N_rays, N_samples)

            # (N_rays, N_samples)
            alpha = 1. - torch.exp(-raw_alpha_reshaped * dists)
            # (N_rays, N_samples) - Calculate transmittance
            weights = alpha * torch.cumprod(torch.cat([torch.ones((N_rays, 1), device=rays_o.device), 1.-alpha + 1e-10], -1), -1)[:, :-1]
            weights = weights + 1e-5 # Add small value to prevent division by zero

            # Normalize weights to form a PDF
            pdf = weights / torch.sum(weights, -1, keepdim=True) # (N_rays, N_samples)
            cdf = torch.cumsum(pdf, -1) # (N_rays, N_samples)
            cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], -1) # (N_rays, N_samples + 1)

        # Inverse transform sampling
        u = torch.rand(N_rays, N_importance, device=rays_o.device)
        u = u.contiguous() # Ensure contiguous memory for searchsorted
        # Find indices of bins where u would fall
        # `right=True` means that a value equal to the rightmost value will be placed to its right
        inds = torch.searchsorted(cdf, u, right=True)

        # Get values of cdf and z_vals at these indices
        # `inds` is from searchsorted on `cdf`, so its range is [0, N_samples]
        # `cdf` has shape (N_rays, N_samples + 1)
        # `z_vals_coarse` has shape (N_rays, N_samples)

        # Indices for cdf (can go up to N_samples, so max index N_samples)
        cdf_below_idx = torch.max(torch.zeros_like(inds - 1), inds - 1)
        cdf_above_idx = torch.min((cdf.shape[-1] - 1) * torch.ones_like(inds), inds)

        # Use gather to get values from cdf
        cdf_g_0 = torch.gather(cdf, -1, cdf_below_idx)
        cdf_g_1 = torch.gather(cdf, -1, cdf_above_idx)

        # Indices for z_vals_coarse (must be clamped to [0, N_samples - 1])
        # z_vals_coarse has N_samples elements, so valid indices are 0 to N_samples - 1
        z_vals_below_idx = torch.clamp(inds - 1, min=0, max=N_samples - 1)
        z_vals_above_idx = torch.clamp(inds, min=0, max=N_samples - 1)

        # Use gather to get values from z_vals_coarse using the clamped indices
        z_vals_g_0 = torch.gather(z_vals_coarse, -1, z_vals_below_idx)
        z_vals_g_1 = torch.gather(z_vals_coarse, -1, z_vals_above_idx)

        # Calculate inverse transform sample points
        denom = cdf_g_1 - cdf_g_0
        denom = torch.where(denom < 1e-5, torch.ones_like(denom), denom) # Prevent division by zero
        t = (u - cdf_g_0) / denom
        z_vals_fine = z_vals_g_0 + t * (z_vals_g_1 - z_vals_g_0)


        # Combine coarse and fine samples and sort them
        z_vals_combined, _ = torch.sort(torch.cat([z_vals_coarse, z_vals_fine], -1), -1)

        # Compute 3D points for combined z_vals
        pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals_combined[..., :, None]
        return pts, z_vals_combined
    else:
        # Only return coarse samples if fine sampling is not requested or not possible
        return pts_coarse, z_vals_coarse

In [ ]:
# --- Test `sample_points_on_rays` function ---

# Dummy camera parameters
H_test, W_test = 10, 10 # Small resolution for quick testing
focal_test = 50.0
c2w_test = torch.eye(4, device=device)
c2w_test[:3, 3] = torch.tensor([0.0, 0.0, -4.0], device=device) # Camera at (0,0,-4)

# Generate rays
test_rays_o, test_rays_d = get_rays(H_test, W_test, focal_test, c2w_test)
N_rays_test = test_rays_o.shape[0]

# Define near and far bounds
near_test, far_test = 2.0, 6.0

# Coarse sampling parameters
N_samples_coarse = 64
N_samples_fine = 128 # Set to 0 to test only coarse sampling

print(f"\nTesting `sample_points_on_rays` with N_rays={N_rays_test}, N_samples_coarse={N_samples_coarse}, N_samples_fine={N_samples_fine}...")

# Test without fine sampling first
pts_coarse_only, z_vals_coarse_only = sample_points_on_rays(
    test_rays_o, test_rays_d, near_test, far_test, N_samples=N_samples_coarse, N_importance=0, perturb=True, lindisp=False,
    coarse_model=None, positional_encoder_pos=None, positional_encoder_dir=None # Explicitly set to None for coarse-only
)

print(f"Shape of points from coarse sampling only: {pts_coarse_only.shape}")
print(f"Shape of depths from coarse sampling only: {z_vals_coarse_only.shape}")
assert pts_coarse_only.shape == (N_rays_test, N_samples_coarse, 3)
assert z_vals_coarse_only.shape == (N_rays_test, N_samples_coarse)
print("Coarse sampling only test successful.")

# Test with both coarse and fine sampling
# Need dummy NeRF model and positional encoders for fine sampling
N_freqs_pos = 10
N_freqs_dir = 4
pos_encoder = PositionalEncoder(N_freqs=N_freqs_pos).to(device)
dir_encoder = PositionalEncoder(N_freqs=N_freqs_dir).to(device) # Use a separate encoder for directions

# Calculate dimensions for NeRFMLP
pos_input_dim = pos_encoder(torch.randn(1, 3).to(device)).shape[-1]
# For view direction, NeRF expects a 3D normalized direction vector for encoding.
dir_input_dim = dir_encoder(torch.randn(1, 3).to(device)).shape[-1] # Encode a 3D vector for view directions

coarse_nerf_model = NeRFMLP(pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]).to(device)

# Provide positional_encoder_dir to the function call
pts_combined, z_vals_combined = sample_points_on_rays(
    test_rays_o, test_rays_d, near_test, far_test, N_samples=N_samples_coarse,
    N_importance=N_samples_fine, perturb=True, lindisp=False,
    coarse_model=coarse_nerf_model, positional_encoder_pos=pos_encoder, positional_encoder_dir=dir_encoder
)

print(f"Shape of points from combined sampling: {pts_combined.shape}")
print(f"Shape of depths from combined sampling: {z_vals_combined.shape}")
assert pts_combined.shape == (N_rays_test, N_samples_coarse + N_samples_fine, 3)
assert z_vals_combined.shape == (N_rays_test, N_samples_coarse + N_samples_fine)
print("Combined (coarse + fine) sampling test successful.")
print("`sample_points_on_rays` function fully tested.")


Testing `sample_points_on_rays` with N_rays=100, N_samples_coarse=64, N_samples_fine=128...
Shape of points from coarse sampling only: torch.Size([100, 64, 3])
Shape of depths from coarse sampling only: torch.Size([100, 64])
Coarse sampling only test successful.
Shape of points from combined sampling: torch.Size([100, 192, 3])
Shape of depths from combined sampling: torch.Size([100, 192])
Combined (coarse + fine) sampling test successful.
`sample_points_on_rays` function fully tested.


**Reasoning**:
The subtask requires developing the `render_rays` function, which implements the volume rendering equation. This function needs to take raw RGB and alpha values, the depths of sampled points, and approximate the continuous integral using numerical quadrature (alpha compositing). I will define this function to perform these calculations.

In [ ]:
import torch
def render_rays(rgb,alpha,z_vals,rays_d):
  """
    Applies the volume rendering equation to composite colors and densities along rays.

    Args:
        rgb (torch.Tensor): Predicted RGB colors for each sampled point (N_rays, N_samples, 3).
        alpha (torch.Tensor): Predicted densities (sigma) for each sampled point (N_rays, N_samples, 1).
        z_vals (torch.Tensor): Depth values of sampled points along rays (N_rays, N_samples).
        rays_d (torch.Tensor): Ray directions (N_rays, 3).

    Returns:
        tuple: A tuple containing:
            - rgb_map (torch.Tensor): Final rendered RGB color for each ray (N_rays, 3).
            - depth_map (torch.Tensor): Final rendered depth for each ray (N_rays, 1).
            - acc_map (torch.Tensor): Accumulated opacity for each ray (N_rays, 1).
            - weights (torch.Tensor): Weights assigned to each sample (N_rays, N_samples, 1).
  """
  dists=z_vals[...,1:]-z_vals[...,:-1]
  dists=torch.cat([dists,torch.tensor([1e10],device=dists.device).expand(dists[...,:1].shape)],-1)
  dists=dists*torch.norm(rays_d[...,None,:],dim=-1)
  alpha=1.-torch.exp(-alpha.squeeze(-1)*dists)
  weights=alpha*torch.cumprod(torch.cat([torch.ones((alpha.shape[0],1),device=alpha.device),1.-alpha+1e-10],-1),-1)[:,:-1]
  rgb_map=torch.sum(weights[...,None]*rgb,-2)
  depth_map=torch.sum(weights*z_vals,-1)
  depth_map=depth_map.unsqueeze(-1)
  acc_map=torch.sum(weights,-1)
  acc_map=acc_map.unsqueeze(-1)
  return rgb_map,depth_map,acc_map,weights.unsqueeze(-1)

import torch

def render_rays(rgb, alpha, z_vals, rays_d):
    
    # Calculate distances between adjacent samples
    dists = z_vals[..., 1:] - z_vals[..., :-1]
    # Append a large distance for the last segment (effectively infinity)
    dists = torch.cat([dists, torch.tensor([1e10], device=dists.device).expand(dists[..., :1].shape)], -1)
    dists = dists * torch.norm(rays_d[..., None, :], dim=-1)

    # Convert predicted sigma (alpha) to opacity (alpha_i)
    # alpha_i = 1 - exp(-sigma_i * delta_i)
    alpha = 1. - torch.exp(-alpha.squeeze(-1) * dists) # (N_rays, N_samples)

    # Calculate transmittance (T_i)
    # T_i = exp(-sum(sigma_j * delta_j)) or T_i = product(1 - alpha_j)
    # The equation used here is: T_i = prod_{j=1}^{i-1} (1 - alpha_j)
    # weights_i = T_i * (1 - exp(-sigma_i * delta_i)) = T_i * alpha_i
    weights = alpha * torch.cumprod(torch.cat([torch.ones((alpha.shape[0], 1), device=alpha.device), 1.-alpha + 1e-10], -1), -1)[:, :-1]

    # Rendered RGB color
    rgb_map = torch.sum(weights[..., None] * rgb, -2) # (N_rays, 3)

    # Rendered Depth map
    depth_map = torch.sum(weights * z_vals, -1) # (N_rays,)
    depth_map = depth_map.unsqueeze(-1) # (N_rays, 1)

    # Accumulated opacity map
    acc_map = torch.sum(weights, -1) # (N_rays,)
    acc_map = acc_map.unsqueeze(-1) # (N_rays, 1)

    return rgb_map, depth_map, acc_map, weights.unsqueeze(-1)

print("Defined the 'render_rays' function for volume rendering.")

**Reasoning**:
Now that the `render_rays` function is defined, I will test it with dummy inputs. This involves generating synthetic `rgb`, `alpha`, `z_vals`, and `rays_d` tensors with appropriate shapes, calling the `render_rays` function, and then asserting that the output shapes match the expected dimensions. This will verify the correctness of the function's logic and tensor operations.

In [ ]:
# --- Test `render_rays` function ---
print("\nTesting `render_rays` function...")

# Dummy parameters matching expected inputs
N_rays_test_render = 10
N_samples_total_test_render = N_samples_coarse + N_samples_fine # e.g., 64 + 128 = 192

# Generate dummy rgb: (N_rays, N_samples, 3) - values between 0 and 1
dummy_rgb = torch.rand(N_rays_test_render, N_samples_total_test_render, 3, device=device)

# Generate dummy alpha: (N_rays, N_samples, 1) - non-negative values
dummy_alpha = torch.rand(N_rays_test_render, N_samples_total_test_render, 1, device=device)

# Generate dummy z_vals: (N_rays, N_samples) - increasing depth values per ray
dummy_z_vals = torch.linspace(near_test, far_test, N_samples_total_test_render, device=device).unsqueeze(0).expand(N_rays_test_render, -1)

# Generate dummy rays_d: (N_rays, 3) - normalized directions
dummy_rays_d = torch.randn(N_rays_test_render, 3, device=device)
dummy_rays_d = dummy_rays_d / torch.norm(dummy_rays_d, dim=-1, keepdim=True)

# Call the render_rays function
rgb_map, depth_map, acc_map, weights = render_rays(dummy_rgb, dummy_alpha, dummy_z_vals, dummy_rays_d)

# Print output shapes
print(f"Output RGB map shape: {rgb_map.shape}")       # Expected: (N_rays, 3)
print(f"Output Depth map shape: {depth_map.shape}")    # Expected: (N_rays, 1)
print(f"Output Accumulated opacity map shape: {acc_map.shape}") # Expected: (N_rays, 1)
print(f"Output Weights shape: {weights.shape}")         # Expected: (N_rays, N_samples, 1)

# Assert output shapes
assert rgb_map.shape == (N_rays_test_render, 3)
assert depth_map.shape == (N_rays_test_render, 1)
assert acc_map.shape == (N_rays_test_render, 1)
assert weights.shape == (N_rays_test_render, N_samples_total_test_render, 1)

print("Test `render_rays` function successful: output shapes are correct.")


Testing `render_rays` function...
Output RGB map shape: torch.Size([10, 3])
Output Depth map shape: torch.Size([10, 1])
Output Accumulated opacity map shape: torch.Size([10, 1])
Output Weights shape: torch.Size([10, 192, 1])
Test `render_rays` function successful: output shapes are correct.


# Task
Integrate all previously defined components (`get_rays`, `sample_points_on_rays`, `NeRFMLP`, `render_rays`, `PositionalEncoder`) into a single high-level function or class that orchestrates the entire NeRF rendering process for a given camera pose to produce an image, involving calling the coarse network, performing hierarchical sampling, and then querying the fine network before applying volume rendering.

## Define Full NeRF Rendering Pipeline

### Subtask:
Integrate all previously defined components (`get_rays`, `sample_points_on_rays`, `NeRFMLP`, `render_rays`, `PositionalEncoder`) into a single high-level function or class that orchestrates the entire NeRF rendering process for a given camera pose to produce an image, involving calling the coarse network, performing hierarchical sampling, and then querying the fine network before applying volume rendering.


**Reasoning**:
The subtask requires defining a high-level `render_image` function that integrates all previously defined NeRF components to orchestrate the entire rendering process. I will implement this function by following the detailed instructions provided, including ray generation, chunk-based processing, coarse sampling, querying the coarse network, volume rendering, hierarchical fine sampling (if enabled), querying the fine network, and combining results. Finally, I will provide a comprehensive test case to verify its functionality and output shapes.



In [ ]:
def render_image(
    H,W,K,c2w,
    nerf_coarse,
    nerf_fine=None,
    N_samples_coarse=64,
    N_importance=0,
    pos_encoder=None,
    dir_encoder=None,
    near=0.,far=1.,
    chunk=1024*32,
    perturb=True,
    lindisp=False,
    white_bkgd=False):
  """
      Orchestrates the entire NeRF rendering process for a given camera pose.

      Args:
          H (int): Image height.
          W (int): Image width.
          K (torch.Tensor): Camera intrinsic matrix.
          c2w (torch.Tensor): Camera-to-world transformation matrix (4x4).
          nerf_coarse (nn.Module): The coarse NeRF MLP model.
          nerf_fine (nn.Module, optional): The fine NeRF MLP model.
          N_samples_coarse (int): Number of samples per ray for the coarse network.
          N_importance (int): Number of importance samples per ray for the fine network.
          pos_encoder (PositionalEncoder): Positional encoder for 3D coordinates.
          dir_encoder (PositionalEncoder): Positional encoder for view directions.
          near (float): Near bound for ray sampling.
          far (float): Far bound for ray sampling.
          chunk (int): The number of rays to process in parallel.
          perturb (bool): Whether to add stochastic perturbation to samples.
          lindisp (bool): If true, sample in inverse depth (disparity) space.
          white_bkgd (bool): If true, use a white background for rendering.

      Returns:
          dict: A dictionary containing rendered RGB, depth, and accumulated opacity maps.
  """
  focal=K[0,0]
  rays_o,rays_d=get_rays(H,W,focal,c2w.to(device))
  rays_o_flat=rays_o.reshape(-1,3)
  rays_d_flat=rays_d.reshape(-1,3)
  N_rays=rays_o_flat.shape[0]
  all_rgb_map_coarse=[]
  all_depth_map_coarse=[]
  all_acc_map_coarse=[]
  all_rgb_map_fine=[]
  all_depth_map_fine=[]
  all_acc_map_fine=[]
  for i in range(0,N_rays,chunk):
    rays_o_chunk=rays_o_flat[i:i+chunk]
    rays_d_chunk=rays_d_flat[i:i+chunk]
    N_rays_chunk=rays_o_chunk.shape[0]
    pts_coarse,z_vals_coarse=sample_points_on_rays(
        rays_o_chunk,rays_d_chunk,near,far,N_samples_coarse,N_importance=0,perturb=perturb,lindisp=lindisp,
        coarse_model=nerf_coarse, positional_encoder_pos=pos_encoder, positional_encoder_dir=dir_encoder
    )
    pts_flat_coarse=pts_coarse.reshape(-1,3)
    encoded_pts_coarse=pos_encoder(pts_flat_coarse)if pos_encoder else pts_flat_coarse
    rays_d_expanded_coarse=rays_d_chunk[:,None,:].expand(-1,N_samples_coarse,-1).reshape(-1,3)
    encoded_dirs_coarse=dir_encoder(rays_d_expanded_coarse) if dir_encoder else rays_d_expanded_coarse
    rgb_coarse,alpha_coarse=nerf_coarse(encoded_pts_coarse,encoded_dirs_coarse)
    rgb_coarse=rgb_coarse.reshape(N_rays_chunk,N_samples_coarse,3)
    alpha_coarse=alpha_coarse.reshape(N_rays_chunk,N_samples_coarse,1)
    rgb_map_coarse,depth_map_coarse,acc_map_coarse,weights_coarse=render_rays(
        rgb_coarse,alpha_coarse,z_vals_coarse,rays_d_chunk
    )
    all_rgb_map_coarse.append(rgb_map_coarse)
    all_depth_map_coarse.append(depth_map_coarse)
    all_acc_map_coarse.append(acc_map_coarse)
    if N_importance>0 and nerf_fine is not None:
      pts_combined,z_vals_combined=sample_points_on_rays(
          rays_o_chunk,rays_d_chunk,near,far,N_samples_coarse,
          N_importance=N_importance,perturb=perturb,lindisp=lindisp,
          coarse_model=nerf_coarse,positional_encoder_pos=pos_encoder,positional_encoder_dir=dir_encoder
      )
      N_total_samples=N_samples_coarse+N_importance
      pts_flat_combined=pts_combined.reshape(-1,3)
      encoded_pts_fine=pos_encoder(pts_flat_combined) if pos_encoder else pts_flat_combined
      rays_d_expanded_fine=rays_d_chunk[:,None,:].expand(-1,N_total_samples,-1).reshape(-1,3)
      encoded_dirs_fine=dir_encoder(rays_d_expanded_fine) if dir_encoder else rays_d_expanded_fine
      rgb_fine,alpha_fine=nerf_fine(encoded_pts_fine,encoded_dirs_fine)
      rgb_fine=rgb_fine.reshape(N_rays_chunk,N_total_samples,3)
      alpha_fine=alpha_fine.reshape(N_rays_chunk,N_total_samples,1)
      rgb_map_fine,depth_map_fine,acc_map_fine,_=render_rays(
          rgb_fine,alpha_fine,z_vals_combined,rays_d_chunk
      )
      all_rgb_map_fine.append(rgb_map_fine)
      all_depth_map_fine.append(depth_map_fine)
      all_acc_map_fine.append(acc_map_fine)
  rgb_map_coarse=torch.cat(all_rgb_map_coarse,0).reshape(H,W,3)
  depth_map_coarse=torch.cat(all_depth_map_coarse,0).reshape(H,W,1)
  acc_map_coarse=torch.cat(all_acc_map_coarse,0).reshape(H,W,1)
  results={
      'rgb_map_coarse':rgb_map_coarse,
      "depth_map_coarse":depth_map_coarse,
      "acc_map_coarse":acc_map_coarse
  }
  if N_importance>0 and nerf_fine is not None:
    rgb_map_fine=torch.cat(all_rgb_map_fine,0).reshape(H,W,3)
    depth_map_fine=torch.cat(all_depth_map_fine,0).reshape(H,W,1)
    acc_map_fine=torch.cat(all_acc_map_fine,0).reshape(H,W,1)
    results['rgb_map_fine']=rgb_map_fine
    results['depth_map_fine']=depth_map_fine
    results['acc_map_fine']=acc_map_fine
    if white_bkgd:
      results['rgb_map_fine']=results['rgb_map_fine']+(1.-results['acc_map_fine'])
      results['rgb_map_fine']=torch.clamp(results['rgb_map_fine'],0.,1.)
  return results

In [ ]:
print("Defined the 'render_image' function encapsulating the NeRF rendering pipeline.")

# --- Test Case for render_image function ---
print("\n--- Starting render_image function tests ---")

# 1a. Instantiate PositionalEncoder for positions and directions
N_freqs_pos = 10
N_freqs_dir = 4
pos_encoder = PositionalEncoder(N_freqs=N_freqs_pos).to(device)
dir_encoder = PositionalEncoder(N_freqs=N_freqs_dir).to(device)

# 1b. Calculate pos_input_dim and dir_input_dim
dummy_coords_for_dim = torch.randn(1, 3).to(device)
encoded_pos_output = pos_encoder(dummy_coords_for_dim)
pos_input_dim = encoded_pos_output.shape[-1]

# For view direction, NeRF expects a 3D normalized direction vector for encoding.
dummy_viewdir_for_dim = torch.randn(1, 3).to(device) # Using 3D vector for directions
encoded_dir_output = dir_encoder(dummy_viewdir_for_dim)
dir_input_dim = encoded_dir_output.shape[-1]

print(f"Calculated pos_input_dim: {pos_input_dim}")
print(f"Calculated dir_input_dim: {dir_input_dim}")

# 1c. Instantiate NeRFMLP for both coarse and fine models
nerf_coarse_model = NeRFMLP(pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]).to(device)
nerf_fine_model = NeRFMLP(pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]).to(device)

print("Instantiated dummy NeRFMLP coarse and fine models.")

# 1d. Define dummy camera parameters and NeRF specific parameters
H_test, W_test = 10, 10 # Small resolution for quick testing
focal_test = 120.0
K_test = torch.tensor([
    [focal_test, 0, W_test/2],
    [0, focal_test, H_test/2],
    [0, 0, 1]
], dtype=torch.float32, device=device)

c2w_test = torch.eye(4, device=device)
c2w_test[:3, 3] = torch.tensor([0.0, 0.0, -4.0], device=device) # Camera at (0,0,-4)

near_test, far_test = 2.0, 6.0
N_samples_coarse_test = 64
N_importance_test = 128
chunk_test = 10 # Small chunk size for testing to ensure chunking logic works

print(f"Test parameters: H={H_test}, W={W_test}, N_samples_coarse={N_samples_coarse_test}, N_importance={N_importance_test}, chunk={chunk_test}")

# 1e. Call the render_image function with these dummy parameters
# Test with both coarse and fine networks
with torch.no_grad():
    results_full_pipeline = render_image(
        H=H_test, W=W_test, K=K_test, c2w=c2w_test,
        nerf_coarse=nerf_coarse_model,
        nerf_fine=nerf_fine_model,
        N_samples_coarse=N_samples_coarse_test,
        N_importance=N_importance_test,
        pos_encoder=pos_encoder,
        dir_encoder=dir_encoder,
        near=near_test, far=far_test,
        chunk=chunk_test,
        perturb=True,
        lindisp=False,
        white_bkgd=True
    )

print("\nResults from full pipeline rendering (coarse + fine):")
# 1f. Assert shapes for full pipeline
expected_image_shape_rgb = (H_test, W_test, 3)
expected_image_shape_mono = (H_test, W_test, 1)

print(f"  rgb_map_coarse shape: {results_full_pipeline['rgb_map_coarse'].shape}")
print(f"  depth_map_coarse shape: {results_full_pipeline['depth_map_coarse'].shape}")
print(f"  acc_map_coarse shape: {results_full_pipeline['acc_map_coarse'].shape}")
assert results_full_pipeline['rgb_map_coarse'].shape == expected_image_shape_rgb
assert results_full_pipeline['depth_map_coarse'].shape == expected_image_shape_mono
assert results_full_pipeline['acc_map_coarse'].shape == expected_image_shape_mono
print("  Coarse map shapes are correct.")

print(f"  rgb_map_fine shape: {results_full_pipeline['rgb_map_fine'].shape}")
print(f"  depth_map_fine shape: {results_full_pipeline['depth_map_fine'].shape}")
print(f"  acc_map_fine shape: {results_full_pipeline['acc_map_fine'].shape}")
assert results_full_pipeline['rgb_map_fine'].shape == expected_image_shape_rgb
assert results_full_pipeline['depth_map_fine'].shape == expected_image_shape_mono
assert results_full_pipeline['acc_map_fine'].shape == expected_image_shape_mono
print("  Fine map shapes are correct.")

print("Full NeRF pipeline rendering test with coarse and fine networks successful.")

# Test with only coarse network
print("\nResults from coarse-only pipeline rendering:")
with torch.no_grad():
    results_coarse_only = render_image(
        H=H_test, W=W_test, K=K_test, c2w=c2w_test,
        nerf_coarse=nerf_coarse_model,
        nerf_fine=None, # Explicitly set to None
        N_samples_coarse=N_samples_coarse_test,
        N_importance=0, # Explicitly set to 0
        pos_encoder=pos_encoder,
        dir_encoder=dir_encoder,
        near=near_test, far=far_test,
        chunk=chunk_test,
        perturb=True,
        lindisp=False,
        white_bkgd=False
    )

print(f"  rgb_map_coarse shape: {results_coarse_only['rgb_map_coarse'].shape}")
print(f"  depth_map_coarse shape: {results_coarse_only['depth_map_coarse'].shape}")
print(f"  acc_map_coarse shape: {results_coarse_only['acc_map_coarse'].shape}")
assert results_coarse_only['rgb_map_coarse'].shape == expected_image_shape_rgb
assert results_coarse_only['depth_map_coarse'].shape == expected_image_shape_mono
assert results_coarse_only['acc_map_coarse'].shape == expected_image_shape_mono
print("  Coarse-only map shapes are correct.")
assert 'rgb_map_fine' not in results_coarse_only # Ensure fine maps are not present
print("Coarse-only NeRF pipeline rendering test successful.")

print("\n--- All render_image tests completed successfully ---")


Defined the 'render_image' function encapsulating the NeRF rendering pipeline.

--- Starting render_image function tests ---
Calculated pos_input_dim: 63
Calculated dir_input_dim: 27
Instantiated dummy NeRFMLP coarse and fine models.
Test parameters: H=10, W=10, N_samples_coarse=64, N_importance=128, chunk=10

Results from full pipeline rendering (coarse + fine):
  rgb_map_coarse shape: torch.Size([10, 10, 3])
  depth_map_coarse shape: torch.Size([10, 10, 1])
  acc_map_coarse shape: torch.Size([10, 10, 1])
  Coarse map shapes are correct.
  rgb_map_fine shape: torch.Size([10, 10, 3])
  depth_map_fine shape: torch.Size([10, 10, 1])
  acc_map_fine shape: torch.Size([10, 10, 1])
  Fine map shapes are correct.
Full NeRF pipeline rendering test with coarse and fine networks successful.

Results from coarse-only pipeline rendering:
  rgb_map_coarse shape: torch.Size([10, 10, 3])
  depth_map_coarse shape: torch.Size([10, 10, 1])
  acc_map_coarse shape: torch.Size([10, 10, 1])
  Coarse-only ma

# Task
## Summary NeRF Implementation

### Q&A
The implemented Neural Radiance Fields (NeRF) model is a PyTorch-based volumetric rendering system capable of synthesizing novel views of a 3D scene from 2D images. It achieves this by representing the scene as a continuous function that maps 3D spatial coordinates and 2D viewing directions to color and density. The core of the implementation leverages a Multi-Layer Perceptron (MLP) for this mapping, enhanced by positional encoding to capture high-frequency details. Hierarchical sampling (coarse and fine networks) is used to efficiently query the MLP and focus computational resources on relevant areas, and the volume rendering equation is applied to accumulate colors and opacities along rays to produce the final pixel values.

### Data Analysis Key Findings
*   **Environment Setup**: Necessary libraries including `torch`, `numpy`, `matplotlib`, `tqdm`, and `scikit-image` were successfully installed.
*   **Positional Encoding**: A `PositionalEncoder` module was implemented, transforming raw input coordinates and view directions into a higher-dimensional feature space using sine and cosine functions. This is crucial for the MLP to capture fine geometric and appearance details.
    *   For 3D coordinates (x,y,z) with `N_freqs=10`, the output dimension is `3 + 2*3*10 = 63`.
    *   For 3D view directions (normalized vectors) with `N_freqs=4`, the output dimension is `3 + 2*3*4 = 27`.
*   **NeRF MLP Architecture**: The `NeRFMLP` class was defined as a deep neural network (8 layers with 256 units by default) with skip connections. It takes positional encoded 3D coordinates to output density and an intermediate feature vector, which is then combined with positional encoded 2D (or 3D normalized) view directions to predict view-dependent RGB color. Sigmoid activation ensures RGB values are between 0 and 1, and ReLU ensures non-negative density.
*   **Ray Generation**: The `get_rays` function was implemented to accurately generate ray origins and directions for each pixel in an image, given camera intrinsics (focal length, image dimensions) and extrinsics (camera-to-world matrix).
*   **Hierarchical Ray Sampling**: The `sample_points_on_rays` function implements both coarse and fine sampling.
    *   **Coarse Sampling**: Uniformly samples `N_samples` points along each ray (with optional perturbation).
    *   **Fine Sampling**: Adaptively samples `N_importance` additional points, biased towards regions of high density, using the coarse network's density predictions to form a PDF for inverse transform sampling. This mechanism efficiently focuses computation on perceptually important regions.
*   **Volume Rendering Equation**: The `render_rays` function approximates the continuous volume rendering integral using numerical quadrature (alpha compositing). It takes predicted RGB colors, densities (alpha), and sample depths (`z_vals`) along a ray to produce the final rendered RGB color, depth, and accumulated opacity for each ray.
*   **Full NeRF Rendering Pipeline**: The `render_image` function integrates all these components into a unified pipeline. It processes rays in chunks, performs coarse sampling, queries the coarse `NeRFMLP`, applies volume rendering, and if fine sampling is enabled, performs hierarchical sampling and queries the fine `NeRFMLP` before a final volume rendering step. It supports options like perturbation, disparity sampling, and white background.

### Insights and Next Steps
*   The implementation successfully established the foundational components of a NeRF model, demonstrating how positional encoding, a deep MLP with skip connections, hierarchical sampling, and the volume rendering equation work together to reconstruct a 3D scene.
*   **Dataset Integration**: The next crucial step is to integrate a real 3D dataset (e.g., Blender dataset or LLFF dataset) for training and validation. This will involve:
    *   Loading camera poses and intrinsic parameters.
    *   Providing ground truth images for training.
*   **Training Loop**: Implement a comprehensive training loop that optimizes the NeRF MLP(s) by minimizing the reconstruction loss (e.g., Mean Squared Error) between the rendered images and ground truth images.
*   **Metrics and Visualization**: Integrate objective evaluation metrics and visualization tools to monitor training progress and qualitatively assess the generated novel views.
*   **Performance Optimization**: Explore techniques such as spatial occupancy grids or other acceleration structures to speed up rendering and training, especially for larger scenes.

## Summarize NeRF Implementation

### Subtask:
Provide a comprehensive summary of the implemented NeRF model, detailing its architecture, the role of positional encoding, hierarchical sampling, the volume rendering equation, and the overall training and rendering process. Highlight key insights and potential next steps for improvement or extension.


## Summary:

### Q&A
The implemented Neural Radiance Fields (NeRF) model is a PyTorch-based volumetric rendering system designed to synthesize novel views of a 3D scene from 2D images. It represents the scene as a continuous function, mapping 3D spatial coordinates and 2D viewing directions to color and density. The core implementation uses a Multi-Layer Perceptron (MLP) enhanced by positional encoding for high-frequency details. Hierarchical sampling (coarse and fine networks) efficiently queries the MLP, while the volume rendering equation accumulates colors and opacities along rays to produce final pixel values.

### Data Analysis Key Findings
*   **Environment Setup**: All necessary libraries such as `torch`, `numpy`, `matplotlib`, `tqdm`, and `scikit-image` were successfully installed.
*   **Positional Encoding**: A `PositionalEncoder` module was implemented to transform input coordinates and view directions into a higher-dimensional feature space using sine and cosine functions. For 3D coordinates with `N_freqs=10`, the output dimension is 63, and for 3D view directions with `N_freqs=4`, the output dimension is 27. This encoding is vital for capturing fine details.
*   **NeRF MLP Architecture**: The `NeRFMLP` class is an 8-layer deep neural network with 256 units per layer and skip connections. It takes positionally encoded 3D coordinates to output density and a feature vector, which, combined with positionally encoded view directions, predicts view-dependent RGB color. Sigmoid activation ensures RGB values are between 0 and 1, and ReLU ensures non-negative density.
*   **Ray Generation**: The `get_rays` function precisely generates ray origins and directions for each pixel using camera intrinsics (focal length, image dimensions) and extrinsics (camera-to-world matrix).
*   **Hierarchical Ray Sampling**: The `sample_points_on_rays` function performs both coarse and fine sampling.
    *   **Coarse Sampling**: Uniformly samples `N_samples` points along each ray.
    *   **Fine Sampling**: Adaptively samples `N_importance` additional points, biased towards high-density regions identified by the coarse network's density predictions.
*   **Volume Rendering Equation**: The `render_rays` function approximates the continuous volume rendering integral using numerical quadrature (alpha compositing). It processes predicted RGB colors, densities, and sample depths to produce the final rendered RGB color, depth, and accumulated opacity for each ray.
*   **Full NeRF Rendering Pipeline**: The `render_image` function integrates all components, processing rays in chunks, performing coarse sampling, querying the coarse `NeRFMLP`, applying volume rendering, and optionally performing fine sampling and querying the fine `NeRFMLP` for a final rendering step.

### Insights or Next Steps
*   The foundational components of a NeRF model have been successfully implemented, demonstrating the integration of positional encoding, a deep MLP, hierarchical sampling, and the volume rendering equation for 3D scene reconstruction.
*   **Dataset Integration and Training Loop**: The immediate next step is to integrate a real 3D dataset (e.g., Blender or LLFF) with corresponding camera poses and ground truth images. This will enable the implementation of a comprehensive training loop to optimize the NeRF MLP(s) by minimizing the reconstruction loss between rendered and ground truth images.
*   **Evaluation and Optimization**: Future work should include integrating objective evaluation metrics and visualization tools to monitor training progress and assess rendering quality. Additionally, exploring performance optimization techniques such as spatial occupancy grids could significantly speed up rendering and training for larger scenes.


# Task
## Prepare Data for NeRF Training

### Subtask:
Load a suitable 3D dataset (e.g., a synthetic Blender dataset or LLFF dataset) including images, camera poses (camera-to-world matrices), and camera intrinsic parameters (focal length, image dimensions). This data will be used to extract rays and their corresponding ground truth pixel values for training. For demonstration purposes, we might need to simulate a small dataset if a real one isn't immediately available.

## Prepare Data for NeRF Training

### Subtask:
Load a suitable 3D dataset (e.g., a synthetic Blender dataset or LLFF dataset) including images, camera poses (camera-to-world matrices), and camera intrinsic parameters (focal length, image dimensions). This data will be used to extract rays and their corresponding ground truth pixel values for training. For demonstration purposes, we might need to simulate a small dataset if a real one isn't immediately available.


In [ ]:
import torch
import numpy as np
import torch.nn.functional as F # Import F for normalize function

# Ensure 'device' is defined for tensor placement
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

# 2. Define dummy image dimensions
H, W = 100, 100
print(f"Dummy image dimensions: H={H}, W={W}")

# 3. Create a list of dummy ground truth images
N_images = 5 # Number of dummy images
dummy_images = []
for _ in range(N_images):
    # Images are typically in [0, 1] range for NeRF training
    dummy_images.append(torch.rand(H, W, 3, device=device))

print(f"Generated {N_images} dummy ground truth images.")

# 4. Generate a list of dummy camera-to-world (c2w) transformation matrices
dummy_c2ws = []
radius = 4.0 # Distance from origin
for i in range(N_images):
    angle = 2 * np.pi * i / N_images # Angle for cameras around a circle

    # Simulate cameras looking towards the origin, positioned in a circle around the y-axis
    # Simplified: camera at (x, 0, z) looking at (0,0,0)
    cam_x = radius * np.sin(angle)
    cam_z = radius * np.cos(angle)
    cam_y = 0.5 # Slightly above the xz-plane

    # Camera position in world coordinates
    position = torch.tensor([cam_x, cam_y, cam_z], dtype=torch.float32, device=device)

    # Simple rotation matrix to make camera look towards origin (0,0,0)
    # Z-axis of camera points towards the origin
    forward_vec = -F.normalize(position, dim=-1) # Direction from camera to origin
    up_vec = torch.tensor([0., 1., 0.], dtype=torch.float32, device=device) # FIXED: Added device=device

    right_vec = F.normalize(torch.cross(up_vec, forward_vec, dim=-1), dim=-1)
    up_vec_corrected = F.normalize(torch.cross(forward_vec, right_vec, dim=-1), dim=-1)


    R = torch.stack([right_vec, up_vec_corrected, forward_vec], dim=1) # (3,3) rotation matrix

    c2w_matrix = torch.eye(4, device=device) # Initialize 4x4 identity
    c2w_matrix[:3, :3] = R # Set rotation
    c2w_matrix[:3, 3] = position # Set translation

    dummy_c2ws.append(c2w_matrix)

print(f"Generated {N_images} dummy camera-to-world matrices.")

# 5. Define a single camera intrinsic matrix K
focal_length = 1.2 * W # Example focal length relative to image width
K = torch.tensor(
    [[focal_length, 0, W / 2],
     [0, focal_length, H / 2],
     [0, 0, 1]],
    dtype=torch.float32,
    device=device
)

print("Defined a single dummy camera intrinsic matrix K.")

# 6. Store the generated data in a structured way
dummy_dataset = {
    'images': dummy_images,
    'c2ws': dummy_c2ws,
    'K': K,
    'H': H,
    'W': W,
    'N_images': N_images
}

print("Stored dummy data in 'dummy_dataset' dictionary.")

# 7. Print verification details
print(f"\nNumber of images in dataset: {dummy_dataset['N_images']}")
print(f"Shape of a sample image: {dummy_dataset['images'][0].shape}")
print(f"Shape of a sample c2w matrix: {dummy_dataset['c2ws'][0].shape}")
print(f"Camera intrinsic matrix K:\n{dummy_dataset['K']}")
print(f"Shape of K: {dummy_dataset['K'].shape}")

Using device: cuda
Dummy image dimensions: H=100, W=100
Generated 5 dummy ground truth images.
Generated 5 dummy camera-to-world matrices.
Defined a single dummy camera intrinsic matrix K.
Stored dummy data in 'dummy_dataset' dictionary.

Number of images in dataset: 5
Shape of a sample image: torch.Size([100, 100, 3])
Shape of a sample c2w matrix: torch.Size([4, 4])
Camera intrinsic matrix K:
tensor([[120.,   0.,  50.],
        [  0., 120.,  50.],
        [  0.,   0.,   1.]], device='cuda:0')
Shape of K: torch.Size([3, 3])


## Define Loss Function and Optimizer for NeRF

### Subtask:
Establish the loss function for training the NeRF model. Typically, the Mean Squared Error (MSE) between the rendered RGB values and the ground truth RGB values is used. Also, define an optimizer (e.g., Adam or AdamW) to update the parameters of both the coarse and fine NeRF MLPs.


**Reasoning**:
I need to define the loss function as `nn.MSELoss` and an optimizer, `optim.Adam`, that includes parameters from both the coarse and fine NeRF models, ensuring proper handling if the fine model is not provided, as specified in the instructions.



In [ ]:
import torch.optim as optim

# 1. Define Loss Function
criterion = torch.nn.MSELoss()

# 2. Define Optimizer
learning_rate = 5e-4 # Example learning rate

# Collect parameters from both coarse and fine models
all_model_params = list(nerf_coarse_model.parameters())
if nerf_fine_model is not None:
    all_model_params.extend(list(nerf_fine_model.parameters()))

optimizer = optim.Adam(all_model_params, lr=learning_rate)

print("Defined Loss Function (MSELoss) and Optimizer (Adam) for NeRF models.")

Defined Loss Function (MSELoss) and Optimizer (Adam) for NeRF models.


In [ ]:
from tqdm.notebook import tqdm
import random
num_epochs=2
N_rays_batch=1024
print(f"Starting NeRF training for {num_epochs} epochs with batch size {N_rays_batch}...")
nerf_coarse_model.to(device)
if nerf_fine_model is not None:
  nerf_fine_model.to(device)
nerf_coarse_model.train()
if nerf_fine_model is not None:
  nerf_fine_model.train()
for epoch in tqdm(range(num_epochs)):
  img_idx=random.randint(0,dummy_dataset['N_images']-1)
  target_img=dummy_dataset['images'][img_idx]
  target_c2w=dummy_dataset['c2ws'][img_idx]
  target_K=dummy_dataset['K']
  H,W=dummy_dataset['H'],dummy_dataset['W']
  focal=target_K[0,0]
  rays_o_all,rays_d_all=get_rays(H,W,focal,target_c2w)
  target_rgb_flat=target_img.reshape(-1,3)
  total_rays_in_image=rays_o_all.shape[0]
  total_batches_per_epoch=(total_rays_in_image+N_rays_batch-1)//N_rays_batch
  pbar=tqdm(range(0,total_rays_in_image,N_rays_batch),desc=f"Epoch {epoch+1}/{num_epochs}")
  for i_batch in pbar:
    rand_idx=torch.randint(0,total_rays_in_image,(N_rays_batch,),device=device)
    batch_rays_o=rays_o_all[rand_idx]
    batch_rays_d=rays_d_all[rand_idx]
    batch_target_rgb=target_rgb_flat[rand_idx]
    rendered_outputs = render_image(
            H=H, W=W, K=target_K, c2w=target_c2w,
            nerf_coarse=nerf_coarse_model,
            nerf_fine=nerf_fine_model,
            N_samples_coarse=N_samples_coarse,
            N_importance=N_samples_fine,
            pos_encoder=pos_encoder,
            dir_encoder=dir_encoder,
            near=near_test, far=far_test,
            chunk=N_rays_batch, # Process the entire batch in one go
            perturb=True,
            lindisp=False,
            white_bkgd=True
        )
    loss=0.0
    if nerf_fine_model is not None:
      rgb_map_fine=rendered_outputs['rgb_map_fine'].reshape(-1,3)
      loss_fine=criterion(rgb_map_fine,batch_target_rgb)
      loss+=loss_fine
      pbar.set_postfix(loss_fine=f'{loss_fine.item():.4f}')
    rgb_map_coarse=rendered_outputs['rgb_map_coarse'].reshape(-1,3)
    loss_coarse=criterion(rgb_map_fine,batch_target_rgb)
    loss+=loss_coarse
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (i_batch // N_rays_batch) % 10 == 0: # Print every 10 full batches processed
      pbar.set_description(f"Epoch {epoch+1}/{num_epochs} Loss: {loss.item():.4f}")

Starting NeRF training for 2 epochs with batch size 1024...


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1/2:   0%|          | 0/10 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 192.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 34.12 MiB is free. Process 2433 has 14.71 GiB memory in use. Of the allocated memory 14.36 GiB is allocated by PyTorch, and 223.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Task
Okay, I will reduce the `N_rays_batch` value to decrease memory usage during training. Then, I will re-run a short training epoch to get the NeRF models into a usable state. This is crucial before we can render any novel views.

```python
# Reduce N_rays_batch to manage memory usage
N_rays_batch = 512 # Reduced from 1024
num_epochs = 1   # Short epoch for quick re-training

print(f"Reducing N_rays_batch to {N_rays_batch} and training for {num_epochs} epoch(s).")
print("Starting NeRF training...")

# Ensure models are on the correct device
nerf_coarse_model.to(device)
if nerf_fine_model is not None:
    nerf_fine_model.to(device)

# Set models to training mode
nerf_coarse_model.train()
if nerf_fine_model is not None:
    nerf_fine_model.train()

# Re-initialize optimizer in case model parameters changed or for fresh state
# (though in this case, parameters haven't changed, but it's good practice for fresh run)
learning_rate = 5e-4 # Example learning rate
all_model_params = list(nerf_coarse_model.parameters())
if nerf_fine_model is not None:
    all_model_params.extend(list(nerf_fine_model.parameters()))
optimizer = optim.Adam(all_model_params, lr=learning_rate)


for epoch in tqdm(range(num_epochs), desc="Overall Training Progress"):
    img_idx = random.randint(0, dummy_dataset['N_images'] - 1)
    target_img = dummy_dataset['images'][img_idx]
    target_c2w = dummy_dataset['c2ws'][img_idx]
    target_K = dummy_dataset['K']
    H, W = dummy_dataset['H'], dummy_dataset['W']
    focal = target_K[0, 0]

    rays_o_all, rays_d_all = get_rays(H, W, focal, target_c2w)
    target_rgb_flat = target_img.reshape(-1, 3)
    total_rays_in_image = rays_o_all.shape[0]

    # Create random indices for batching rays
    rand_idx_all = torch.randperm(total_rays_in_image, device=device)

    # Iterate through batches
    num_batches = (total_rays_in_image + N_rays_batch - 1) // N_rays_batch
    pbar = tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{num_epochs} Ray Batches")

    for i_batch in pbar:
        start_idx = i_batch * N_rays_batch
        end_idx = min((i_batch + 1) * N_rays_batch, total_rays_in_image)
        current_batch_size = end_idx - start_idx

        # Select a random subset of rays for this batch from the pre-shuffled indices
        batch_indices = rand_idx_all[start_idx:end_idx]

        batch_rays_o = rays_o_all[batch_indices]
        batch_rays_d = rays_d_all[batch_indices]
        batch_target_rgb = target_rgb_flat[batch_indices]

        # Render image for the current batch of rays
        rendered_outputs = render_image(
            H=H, W=W, K=target_K, c2w=target_c2w,
            nerf_coarse=nerf_coarse_model,
            nerf_fine=nerf_fine_model,
            N_samples_coarse=N_samples_coarse,
            N_importance=N_samples_fine,
            pos_encoder=pos_encoder,
            dir_encoder=dir_encoder,
            near=near_test, far=far_test,
            chunk=current_batch_size, # Process the entire selected batch in one go
            perturb=True,
            lindisp=False,
            white_bkgd=True
        )

        loss = 0.0
        # Compute loss for fine network if available
        if nerf_fine_model is not None:
            # Note: render_image returns H,W,3 or H,W,1. Need to reshape for loss calculation with batch_target_rgb
            rgb_map_fine = rendered_outputs['rgb_map_fine'].reshape(-1, 3)
            loss_fine = criterion(rgb_map_fine, batch_target_rgb)
            loss += loss_fine
            pbar.set_postfix(loss_fine=f'{loss_fine.item():.4f}')
        
        # Compute loss for coarse network
        rgb_map_coarse = rendered_outputs['rgb_map_coarse'].reshape(-1, 3)
        # It's typical to sum the loss from both coarse and fine, or just fine.
        # Here, we'll sum both to encourage coarse network learning as well.
        loss_coarse = criterion(rgb_map_coarse, batch_target_rgb)
        loss += loss_coarse

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i_batch + 1) % 50 == 0: # Print update more frequently for short runs
            pbar.set_description(f"Epoch {epoch+1}/{num_epochs} Batch {i_batch+1}/{num_batches} Loss: {loss.item():.4f}")

print("\n--- Short NeRF Training Complete ---")
```

## Reduce Ray Batch Size and Retrain (Short Epoch)

### Subtask:
Reduce the `N_rays_batch` value to decrease the memory usage per batch during training. Then, re-run the training loop for a single epoch to get the models into a usable state.


In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.notebook import tqdm
import random

# Ensure 'device' is defined for tensor placement
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Class and function definitions from previous cells to ensure self-contained block ---

class PositionalEncoder(nn.Module):
  def __init__(self, N_freqs, log_space=True):
    super().__init__()
    self.N_freqs = N_freqs
    self.log_space = log_space
    if log_space:
      self.freq_bands = 2.**torch.linspace(0., N_freqs - 1, N_freqs) * torch.pi
    else:
      self.freq_bands = torch.linspace(1., 2.**(N_freqs - 1), N_freqs) * torch.pi

  def forward(self, x):
    x_expanded = x.unsqueeze(-1)
    x_modulated = x_expanded * self.freq_bands.to(x.device)
    sin_encoded = torch.sin(x_modulated)
    cos_encoded = torch.cos(x_modulated)
    sin_encoded = sin_encoded.flatten(start_dim=-2)
    cos_encoded = cos_encoded.flatten(start_dim=-2)
    return torch.cat([x, sin_encoded, cos_encoded], dim=-1)

class NeRFMLP(nn.Module):
  def __init__(self, pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]):
    super().__init__()
    self.D = D
    self.W = W
    self.skips = skips

    self.pts_linears = nn.ModuleList([nn.Linear(pos_input_dim, W)])

    for i in range(1, D):
        layer_in_dim = W
        if i in self.skips:
            layer_in_dim += pos_input_dim
        self.pts_linears.append(nn.Linear(layer_in_dim, W))

    self.alpha_linear = nn.Linear(W, 1)
    self.feature_linear = nn.Linear(W, W)

    self.views_linears = nn.ModuleList([nn.Linear(W + dir_input_dim, W // 2)])
    self.views_linears.extend([nn.Linear(W // 2, W // 2) for _ in range(2)])
    self.rgb_linear = nn.Linear(W // 2, 3)

  def forward(self, x, d):
    h = x

    for i, l in enumerate(self.pts_linears):
      if i in self.skips:
        h = torch.cat([x, h], -1)

      h = l(h)
      h = F.relu(h)

    alpha = self.alpha_linear(h)
    feature = self.feature_linear(h)

    h_views = torch.cat([feature, d], -1)
    for i, l in enumerate(self.views_linears):
      h_views = l(h_views)
      h_views = F.relu(h_views)
    rgb = self.rgb_linear(h_views)

    rgb = torch.sigmoid(rgb)
    alpha = F.relu(alpha)

    return rgb, alpha

def get_rays(H, W, focal, c2w):
  i, j = torch.meshgrid(
      torch.linspace(0, W - 1, W, device=c2w.device),
      torch.linspace(0, H - 1, H, device=c2w.device),
      indexing='xy'
  )
  dirs = torch.stack([
      (i - W * 0.5) / focal,
      -(j - H * 0.5) / focal,
      -torch.ones_like(i)
  ], -1)
  rays_d = torch.sum(dirs[..., None, :] * c2w[:3, :3], -1)
  rays_o = c2w[:3, 3].expand(rays_d.shape)
  rays_o = rays_o.reshape(-1, 3)
  rays_d = rays_d.reshape(-1, 3)
  return rays_o, rays_d

def sample_points_on_rays(rays_o, rays_d, near, far, N_samples, N_importance=0, perturb=True,
                          lindisp=False, coarse_model=None, positional_encoder_pos=None, positional_encoder_dir=None):
    N_rays = rays_o.shape[0]

    t_vals = torch.linspace(0., 1., N_samples, device=rays_o.device)
    if lindisp:
        t_vals = 1. / (1. / near * (1. - t_vals) + 1. / far * t_vals)
    else:
        t_vals = near * (1. - t_vals) + far * t_vals

    if perturb:
        mids = .5 * (t_vals[..., 1:] + t_vals[..., :-1])
        upper = torch.cat([mids, t_vals[..., -1:]], -1)
        lower = torch.cat([t_vals[..., :1], mids], -1)
        t_rand = torch.rand(N_samples, device=rays_o.device)
        t_vals = lower + (upper - lower) * t_rand

    pts_coarse = rays_o[..., None, :] + rays_d[..., None, :] * t_vals[..., :, None]
    z_vals_coarse = t_vals.expand(N_rays, N_samples)

    if N_importance > 0 and coarse_model is not None and positional_encoder_pos is not None and positional_encoder_dir is not None:
        with torch.no_grad():
            pts_flat = pts_coarse.reshape(-1, 3)
            encoded_pts_flat = positional_encoder_pos(pts_flat)
            rays_d_expanded = rays_d[..., None, :].expand(-1, N_samples, -1).reshape(-1, 3)
            encoded_rays_d_expanded = positional_encoder_dir(rays_d_expanded)
            _, raw_alpha = coarse_model(encoded_pts_flat, encoded_rays_d_expanded)

            dists = torch.cat([z_vals_coarse[..., 1:] - z_vals_coarse[..., :-1],
                               torch.tensor([1e10], device=rays_o.device).expand(z_vals_coarse[..., :1].shape)], -1)

            raw_alpha_reshaped = raw_alpha.reshape(N_rays, N_samples)

            alpha = 1. - torch.exp(-raw_alpha_reshaped * dists)
            weights = alpha * torch.cumprod(torch.cat([torch.ones((N_rays, 1), device=rays_o.device), 1.-alpha + 1e-10], -1), -1)[:, :-1]
            weights = weights + 1e-5

            pdf = weights / torch.sum(weights, -1, keepdim=True)
            cdf = torch.cumsum(pdf, -1)
            cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], -1)

        u = torch.rand(N_rays, N_importance, device=rays_o.device)
        u = u.contiguous()
        inds = torch.searchsorted(cdf, u, right=True)

        cdf_below_idx = torch.max(torch.zeros_like(inds - 1), inds - 1)
        cdf_above_idx = torch.min((cdf.shape[-1] - 1) * torch.ones_like(inds), inds)

        cdf_g_0 = torch.gather(cdf, -1, cdf_below_idx)
        cdf_g_1 = torch.gather(cdf, -1, cdf_above_idx)

        z_vals_below_idx = torch.clamp(inds - 1, min=0, max=N_samples - 1)
        z_vals_above_idx = torch.clamp(inds, min=0, max=N_samples - 1)

        z_vals_g_0 = torch.gather(z_vals_coarse, -1, z_vals_below_idx)
        z_vals_g_1 = torch.gather(z_vals_coarse, -1, z_vals_above_idx)

        denom = cdf_g_1 - cdf_g_0
        denom = torch.where(denom < 1e-5, torch.ones_like(denom), denom)
        t = (u - cdf_g_0) / denom
        z_vals_fine = z_vals_g_0 + t * (z_vals_g_1 - z_vals_g_0)

        z_vals_combined, _ = torch.sort(torch.cat([z_vals_coarse, z_vals_fine], -1), -1)

        pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals_combined[..., :, None]
        return pts, z_vals_combined
    else:
        return pts_coarse, z_vals_coarse

def render_rays(rgb, alpha, z_vals, rays_d):
    dists = z_vals[..., 1:] - z_vals[..., :-1]
    dists = torch.cat([dists, torch.tensor([1e10], device=dists.device).expand(dists[..., :1].shape)], -1)
    dists = dists * torch.norm(rays_d[..., None, :], dim=-1)

    alpha = 1. - torch.exp(-alpha.squeeze(-1) * dists)
    weights = alpha * torch.cumprod(torch.cat([torch.ones((alpha.shape[0], 1), device=alpha.device), 1.-alpha + 1e-10], -1), -1)[:, :-1]

    rgb_map = torch.sum(weights[..., None] * rgb, -2)

    depth_map = torch.sum(weights * z_vals, -1)
    depth_map = depth_map.unsqueeze(-1)

    acc_map = torch.sum(weights, -1)
    acc_map = acc_map.unsqueeze(-1)

    return rgb_map, depth_map, acc_map, weights.unsqueeze(-1)

def render_image(
    H,W,K,c2w,
    nerf_coarse,
    nerf_fine=None,
    N_samples_coarse=64,
    N_importance=0,
    pos_encoder=None,
    dir_encoder=None,
    near=0.,far=1.,
    chunk=1024*32, # This chunk refers to the maximum number of rays processed by render_image
    perturb=True,
    lindisp=False,
    white_bkgd=False,
    chunk_mlp=1024): # New parameter for chunking MLP calls

  focal=K[0,0]
  rays_o,rays_d=get_rays(H,W,focal,c2w.to(device))
  rays_o_flat=rays_o.reshape(-1,3)
  rays_d_flat=rays_d.reshape(-1,3)
  N_rays=rays_o_flat.shape[0]
  all_rgb_map_coarse=[]
  all_depth_map_coarse=[]
  all_acc_map_coarse=[]
  all_rgb_map_fine=[]
  all_depth_map_fine=[]
  all_acc_map_fine=[]

  # Process rays in smaller chunks, controlled by the 'chunk' parameter
  for i in range(0,N_rays,chunk):
    rays_o_chunk=rays_o_flat[i:i+chunk]
    rays_d_chunk=rays_d_flat[i:i+chunk]
    N_rays_chunk=rays_o_chunk.shape[0]

    pts_coarse,z_vals_coarse=sample_points_on_rays(
        rays_o_chunk,rays_d_chunk,near,far,N_samples_coarse,N_importance=0,perturb=perturb,lindisp=lindisp,
        coarse_model=nerf_coarse, positional_encoder_pos=pos_encoder, positional_encoder_dir=dir_encoder
    )

    pts_flat_coarse=pts_coarse.reshape(-1,3)
    N_points_coarse = pts_flat_coarse.shape[0] # N_rays_chunk * N_samples_coarse

    # Apply chunking for MLP calls within the current rays_chunk
    raw_rgb_coarse = []
    raw_alpha_coarse = []
    for j in range(0, N_points_coarse, chunk_mlp):
        pts_batch = pts_flat_coarse[j:j+chunk_mlp]

        # Calculate corresponding ray indices for this batch of points
        # Each point in pts_flat_coarse is (ray_idx * N_samples_coarse + sample_idx)
        # So, ray_idx = (current_point_global_idx) // N_samples_coarse
        ray_indices_for_batch = torch.div(torch.arange(j, j + pts_batch.shape[0], device=device), N_samples_coarse, rounding_mode='floor')
        dirs_batch = rays_d_chunk[ray_indices_for_batch]

        encoded_pts = pos_encoder(pts_batch) if pos_encoder else pts_batch
        encoded_dirs = dir_encoder(dirs_batch) if dir_encoder else dirs_batch

        rgb_chunk, alpha_chunk = nerf_coarse(encoded_pts, encoded_dirs)
        raw_rgb_coarse.append(rgb_chunk)
        raw_alpha_coarse.append(alpha_chunk)

    rgb_coarse = torch.cat(raw_rgb_coarse, 0).reshape(N_rays_chunk,N_samples_coarse,3)
    alpha_coarse = torch.cat(raw_alpha_coarse, 0).reshape(N_rays_chunk,N_samples_coarse,1)

    rgb_map_coarse,depth_map_coarse,acc_map_coarse,weights_coarse=render_rays(
        rgb_coarse,alpha_coarse,z_vals_coarse,rays_d_chunk
    )
    all_rgb_map_coarse.append(rgb_map_coarse)
    all_depth_map_coarse.append(depth_map_coarse)
    all_acc_map_coarse.append(acc_map_coarse)

    if N_importance > 0 and nerf_fine is not None:
      pts_combined,z_vals_combined=sample_points_on_rays(
          rays_o_chunk,rays_d_chunk,near,far,N_samples_coarse,
          N_importance=N_importance,perturb=perturb,lindisp=lindisp,
          coarse_model=nerf_coarse,positional_encoder_pos=pos_encoder,positional_encoder_dir=dir_encoder
      )
      N_total_samples=N_samples_coarse+N_importance
      pts_flat_combined=pts_combined.reshape(-1,3)
      N_points_fine = pts_flat_combined.shape[0] # N_rays_chunk * N_total_samples

      # Apply chunking for fine MLP calls
      raw_rgb_fine = []
      raw_alpha_fine = []
      for j in range(0, N_points_fine, chunk_mlp):
          pts_batch = pts_flat_combined[j:j+chunk_mlp]

          # Calculate corresponding ray indices for this batch of points
          ray_indices_for_batch = torch.div(torch.arange(j, j + pts_batch.shape[0], device=device), N_total_samples, rounding_mode='floor')
          dirs_batch = rays_d_chunk[ray_indices_for_batch]

          encoded_pts = pos_encoder(pts_batch) if pos_encoder else pts_batch
          encoded_dirs = dir_encoder(dirs_batch) if dir_encoder else dirs_batch

          rgb_chunk, alpha_chunk = nerf_fine(encoded_pts, encoded_dirs)
          raw_rgb_fine.append(rgb_chunk)
          raw_alpha_fine.append(alpha_chunk)

      rgb_fine=torch.cat(raw_rgb_fine, 0).reshape(N_rays_chunk,N_total_samples,3)
      alpha_fine=torch.cat(raw_alpha_fine, 0).reshape(N_rays_chunk,N_total_samples,1)

      rgb_map_fine,depth_map_fine,acc_map_fine,_=render_rays(
          rgb_fine,alpha_fine,z_vals_combined,rays_d_chunk
      )
      all_rgb_map_fine.append(rgb_map_fine)
      all_depth_map_fine.append(depth_map_fine)
      all_acc_map_fine.append(acc_map_fine)

  rgb_map_coarse=torch.cat(all_rgb_map_coarse,0).reshape(H,W,3)
  depth_map_coarse=torch.cat(all_depth_map_coarse,0).reshape(H,W,1)
  acc_map_coarse=torch.cat(all_acc_map_coarse,0).reshape(H,W,1)

  results={
      'rgb_map_coarse':rgb_map_coarse,
      "depth_map_coarse":depth_map_coarse,
      "acc_map_coarse":acc_map_coarse
  }

  if N_importance > 0 and nerf_fine is not None:
    rgb_map_fine=torch.cat(all_rgb_map_fine,0).reshape(H,W,3)
    depth_map_fine=torch.cat(all_depth_map_fine,0).reshape(H,W,1)
    acc_map_fine=torch.cat(all_acc_map_fine,0).reshape(H,W,1)
    results['rgb_map_fine']=rgb_map_fine
    results['depth_map_fine']=depth_map_fine
    results['acc_map_fine']=acc_map_fine
    if white_bkgd:
      results['rgb_map_fine']=results['rgb_map_fine']+(1.-results['acc_map_fine'])
      results['rgb_map_fine']=torch.clamp(results['rgb_map_fine'],0.,1.)
  return results


# --- Start of new training loop implementation ---

# 1. Set N_rays_batch to 256 and num_epochs to 1
N_rays_batch = 256
num_epochs = 1
# The 'chunk' parameter for render_image controls the ray batch size processed by render_image.
chunk = 64 # Use a smaller chunk to break down rays within render_image
chunk_mlp = 1024 # This new chunk is for internal MLP calls (points)

print(f"Reducing N_rays_batch to {N_rays_batch} and training for {num_epochs} epoch.")

# 2. Ensure models are on the appropriate device and in training mode
# Re-initialize positional encoders as well to ensure they are available
N_freqs_pos = 10
N_freqs_dir = 4
pos_encoder = PositionalEncoder(N_freqs=N_freqs_pos).to(device)
dir_encoder = PositionalEncoder(N_freqs=N_freqs_dir).to(device)

# Calculate input dimensions for NeRFMLP
dummy_coords_for_dim = torch.randn(1, 3).to(device)
pos_input_dim = pos_encoder(dummy_coords_for_dim).shape[-1]
dummy_viewdir_for_dim = torch.randn(1, 3).to(device) # Use 3D vector for directions
dir_input_dim = dir_encoder(dummy_viewdir_for_dim).shape[-1]

# Re-instantiate NeRFMLP for both coarse and fine models if not already present or needs reset
# (Assuming nerf_coarse_model and nerf_fine_model are defined from previous cell '97fb8531')
if 'nerf_coarse_model' not in globals() or 'nerf_fine_model' not in globals():
    nerf_coarse_model = NeRFMLP(pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]).to(device)
    nerf_fine_model = NeRFMLP(pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]).to(device)
    print("Re-instantiated NeRFMLP coarse and fine models.")

nerf_coarse_model.to(device).train()
if nerf_fine_model is not None:
  nerf_fine_model.to(device).train()

# 3. Re-initialize the optimizer
learning_rate = 5e-4 # Example learning rate
all_model_params = list(nerf_coarse_model.parameters())
if nerf_fine_model is not None:
    all_model_params.extend(list(nerf_fine_model.parameters()))
optimizer = optim.Adam(all_model_params, lr=learning_rate)

print("Optimizer re-initialized.")

# Assuming dummy_dataset is available from previous cells.
N_samples_coarse = 64 # from previous cells
N_importance = 128 # from previous cells (N_samples_fine from before)
near_test, far_test = 2.0, 6.0 # from previous cells

print(f"Starting NeRF training for {num_epochs} epoch with batch size {N_rays_batch}...")

# 4. Implement the training loop
for epoch in tqdm(range(num_epochs), desc="Epoch Loop"):
    img_idx = random.randint(0, dummy_dataset['N_images'] - 1)
    target_img = dummy_dataset['images'][img_idx]
    target_c2w = dummy_dataset['c2ws'][img_idx]
    target_K = dummy_dataset['K']
    H, W = dummy_dataset['H'], dummy_dataset['W']
    focal = target_K[0, 0]

    rays_o_all, rays_d_all = get_rays(H, W, focal, target_c2w)
    target_rgb_flat = target_img.reshape(-1, 3)
    total_rays_in_image = rays_o_all.shape[0]

    # Create a shuffled list of indices for all rays
    rand_idx_all = torch.randperm(total_rays_in_image, device=device)

    for i in tqdm(range(0, total_rays_in_image, N_rays_batch), desc=f"Batch Loop (Epoch {epoch+1})"):
        current_batch_size_training = min(N_rays_batch, total_rays_in_image - i)
        if current_batch_size_training == 0:
            continue

        batch_indices = rand_idx_all[i : i + current_batch_size_training]
        batch_rays_o = rays_o_all[batch_indices]
        batch_rays_d = rays_d_all[batch_indices]
        batch_target_rgb = target_rgb_flat[batch_indices]

        rendered_outputs = render_image(
                H=H, W=W, K=target_K, c2w=target_c2w,
                nerf_coarse=nerf_coarse_model,
                nerf_fine=nerf_fine_model,
                N_samples_coarse=N_samples_coarse,
                N_importance=N_importance,
                pos_encoder=pos_encoder,
                dir_encoder=dir_encoder,
                near=near_test, far=far_test,
                chunk=current_batch_size_training, # Use the batch size for render_image's outer loop
                perturb=True,
                lindisp=False,
                white_bkgd=True,
                chunk_mlp=chunk_mlp # Pass the new MLP chunk size
            )

        loss = 0.0
        if nerf_fine_model is not None:
            rgb_map_fine = rendered_outputs['rgb_map_fine'].reshape(-1, 3)
            loss_fine = criterion(rgb_map_fine, batch_target_rgb)
            loss += loss_fine

        # Always calculate coarse loss
        rgb_map_coarse = rendered_outputs['rgb_map_coarse'].reshape(-1, 3)
        loss_coarse = criterion(rgb_map_coarse, batch_target_rgb)
        loss += loss_coarse

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i // N_rays_batch) % 10 == 0:
            tqdm.write(f"Epoch {epoch+1}/{num_epochs}, Batch {i//N_rays_batch+1}/{ (total_rays_in_image + N_rays_batch - 1) // N_rays_batch}, Loss: {loss.item():.4f}")

print("NeRF training complete for specified epoch.")

Reducing N_rays_batch to 256 and training for 1 epoch.
Optimizer re-initialized.
Starting NeRF training for 1 epoch with batch size 256...


Epoch Loop:   0%|          | 0/1 [00:00<?, ?it/s]

Batch Loop (Epoch 1):   0%|          | 0/40 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2433 has 14.74 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 55.81 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Task
## Generate Novel Camera Pose

### Subtask:
Define a new `camera-to-world` transformation matrix (`c2w`) that represents a novel viewing angle not seen during the training data. This will involve calculating a camera position and orientation.

### Reasoning:
To generate a novel view, a new `c2w` matrix is required. This matrix defines the camera's position and orientation in the 3D world. I will calculate a new camera position (e.g., slightly offset from the training path) and then derive its rotation to ensure it still looks towards the origin, similar to how the training `c2w` matrices were generated. This will provide a `c2w` matrix representing a perspective not present in the training data.

## Generate Novel Camera Pose

### Subtask:
Define a new `camera-to-world` transformation matrix (`c2w`) that represents a novel viewing angle not seen during the training data. This will involve calculating a camera position and orientation.


**Reasoning**:
The subtask requires defining a new camera-to-world transformation matrix (`c2w`) for a novel viewing angle. I will implement the detailed instructions to calculate the camera's new position and orientation and construct the `novel_c2w` matrix.



In [ ]:
import torch
import numpy as np
import torch.nn.functional as F

# Ensure 'device' is defined
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reuse parameters from dummy_dataset generation
# H, W, K, and radius were defined in the dummy_dataset generation cell (4051a1a0)
# We need the 'radius' variable, if not in globals, we'll re-define it based on previous context.
if 'radius' not in globals():
    # This value was 4.0 in the previous dummy data generation
    radius = 4.0
    print(f"'radius' not found in globals, re-initializing to {radius}")

print("Generating novel camera pose...")

# 1. Define a new_angle for the camera
# Example: halfway between the first two training angles (0 and 2*pi/N_images)
# Assuming N_images was 5 from previous context, angle was 2*pi*i/5
# So, 0, 2*pi/5, 4*pi/5, 6*pi/5, 8*pi/5
# A novel angle could be pi/5 (halfway between 0 and 2*pi/5)
# Let's pick an angle not used in the 5 images.
# For N_images=5, angles were 0, 0.4*pi, 0.8*pi, 1.2*pi, 1.6*pi
# Let's choose an angle not exactly on these points, e.g., (0.2 * pi)
new_angle = 0.2 * np.pi # Example: 36 degrees, halfway between angle 0 and angle 1

# 2. Define a new_cam_y position
# Original cam_y was 0.5. Let's make it slightly different, e.g., 0.6
new_cam_y = 0.6

# 3. Calculate the new cam_x and cam_z coordinates
new_cam_x = radius * np.sin(new_angle)
new_cam_z = radius * np.cos(new_angle)

# 4. Construct the position tensor for the new camera
position = torch.tensor([new_cam_x, new_cam_y, new_cam_z], dtype=torch.float32, device=device)

# 5. Calculate the forward_vec (camera looks towards the origin)
forward_vec = -F.normalize(position, dim=-1)

# 6. Define an up_vec
up_vec = torch.tensor([0., 1., 0.], dtype=torch.float32, device=device)

# 7. Calculate the right_vec
right_vec = F.normalize(torch.cross(up_vec, forward_vec, dim=-1), dim=-1)

# 8. Calculate the up_vec_corrected
up_vec_corrected = F.normalize(torch.cross(forward_vec, right_vec, dim=-1), dim=-1)

# 9. Create the rotation matrix R
R = torch.stack([right_vec, up_vec_corrected, forward_vec], dim=1)

# 10. Initialize a 4x4 identity matrix novel_c2w
novel_c2w = torch.eye(4, device=device)

# 11. Set the top-left 3x3 block of novel_c2w to the calculated rotation matrix R
novel_c2w[:3, :3] = R

# 12. Set the translation part (top-right 3x1 column) of novel_c2w to the calculated position
novel_c2w[:3, 3] = position

# 13. Print the resulting novel_c2w matrix
print(f"Novel camera-to-world matrix (novel_c2w):\n{novel_c2w}")

print("Novel camera pose (c2w matrix) generated successfully.")

Generating novel camera pose...
Novel camera-to-world matrix (novel_c2w):
tensor([[-0.8090, -0.0872, -0.5813,  2.3511],
        [ 0.0000,  0.9889, -0.1483,  0.6000],
        [ 0.5878, -0.1200, -0.8001,  3.2361],
        [ 0.0000,  0.0000,  0.0000,  1.0000]], device='cuda:0')
Novel camera pose (c2w matrix) generated successfully.


## Final Task

### Subtask:
Summarize the process of generating a novel camera pose for rendering.


## Summary:

### Q&A
The task was to define a new `camera-to-world` transformation matrix (`c2w`) that represents a novel viewing angle. This was successfully completed by calculating a new camera position and orientation, resulting in a `novel_c2w` matrix.

### Data Analysis Key Findings
*   The `radius` parameter, crucial for determining camera distance from the origin, was initialized to `4.0`, and computations were performed on the `cuda` device.
*   A novel camera viewing angle was defined as `0.2 * \pi` (approximately 36 degrees), and the `y`-coordinate for the camera position was set to `0.6`.
*   The `x` and `z` coordinates for the novel camera position were calculated as `radius * sin(new_angle)` and `radius * cos(new_angle)` respectively, resulting in a `position` tensor representing the camera's location.
*   The camera's orientation was derived by calculating a `forward_vec` pointing towards the origin, an `up_vec`, and a `right_vec`, which were then used to construct the 3x3 rotation matrix `R`.
*   A 4x4 `novel_c2w` transformation matrix was successfully constructed by embedding the calculated `R` into its top-left 3x3 block and the `position` tensor into its translation component.

### Insights or Next Steps
*   The generated `novel_c2w` matrix can now be used as input to a rendering pipeline to synthesize an image from this previously unseen viewpoint.
*   The methodology can be extended to generate a sequence of novel camera poses, enabling the creation of a smooth fly-through animation or a 360-degree view around the object.


## Render Novel View

### Subtask:
Use the `render_image` function with the (shortly) trained NeRF models and the newly generated camera pose (`novel_c2w`) to render an image from this novel viewpoint. This will leverage the coarse and fine networks and volume rendering.


**Reasoning**:
The subtask requires rendering a novel view using the `render_image` function with the trained NeRF models and the `novel_c2w` camera pose. I will set the models to evaluation mode, call `render_image` within a `torch.no_grad()` block, and store the results.



In [ ]:
print("\n--- Rendering novel view ---")

# 1. Set NeRF models to evaluation mode
nerf_coarse_model.eval()
if nerf_fine_model is not None:
    nerf_fine_model.eval()

# 2. Call the render_image function with the novel_c2w
with torch.no_grad():
    novel_view_results = render_image(
        H=dummy_dataset['H'], W=dummy_dataset['W'], K=dummy_dataset['K'], c2w=novel_c2w,
        nerf_coarse=nerf_coarse_model,
        nerf_fine=nerf_fine_model,
        N_samples_coarse=N_samples_coarse,
        N_importance=N_importance,
        pos_encoder=pos_encoder,
        dir_encoder=dir_encoder,
        near=near_test, far=far_test,
        chunk=chunk, # Use the reduced chunk size for outer loop of render_image
        perturb=False, # No perturbation during rendering
        lindisp=False,
        white_bkgd=True,
        chunk_mlp=chunk_mlp # Use the reduced chunk size for MLP calls
    )

print("Novel view rendered successfully.")

# Optionally, print shapes of the rendered outputs
print(f"\nRendered novel view (fine) RGB map shape: {novel_view_results['rgb_map_fine'].shape}")
print(f"Rendered novel view (fine) Depth map shape: {novel_view_results['depth_map_fine'].shape}")


--- Rendering novel view ---


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2433 has 14.74 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 55.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Task
An `OutOfMemoryError` occurred during the rendering of the novel view, indicating that the current model and data parameters are too large for the available GPU memory, even with the existing chunking strategy. This also happened during the preceding training run.

To resolve this, I will drastically reduce the parameters of the NeRF model and the data being processed. This includes reducing image resolution, the number of samples per ray, the NeRF MLP width and depth, and positional encoding frequencies. After re-initializing the models with these smaller parameters, I will perform a very short training run and then attempt to render and visualize the novel view again.

**Plan:**

1.  **Reduce NeRF Model and Data Parameters**:
    *   Set `H, W = 20, 20` for image dimensions.
    *   Set `N_images = 3` for the dummy dataset.
    *   Set `N_samples_coarse = 16` and `N_importance = 16` for ray sampling.
    *   Set `N_freqs_pos = 4` and `N_freqs_dir = 2` for positional encoding.
    *   Set `W_mlp = 64` (MLP width) and `D_mlp = 4` (MLP depth).
    *   Adjust `focal_length` accordingly.
2.  **Re-initialize Data and Models**:
    *   Re-create the `dummy_dataset` with the new `H, W, N_images`, and `focal_length`.
    *   Re-instantiate `PositionalEncoder` and `NeRFMLP` (coarse and fine) with the new `N_freqs` and `W_mlp, D_mlp` values.
3.  **Perform a short training run**:
    *   Set `num_epochs = 1`.
    *   Set `N_rays_batch = 128` (outer training loop).
    *   Set `chunk = 32` (for `render_image` ray chunking).
    *   Set `chunk_mlp = 256` (for `render_image` MLP point chunking).
    *   Use the new `nerf_coarse_model`, `nerf_fine_model`, `pos_encoder`, `dir_encoder`.
4.  **Generate Novel Camera Pose**: Re-run the `novel_c2w` generation using the updated `radius` if applicable.
5.  **Render Novel View**: Call the `render_image` function with the newly trained (small) models and the `novel_c2w`.
6.  **Visualize Generated Image**: Display the rendered image using `matplotlib`.
7.  **Final Task**: Summarize the process of completing a short training run, generating a novel view, and visualizing the rendered image from the NeRF model.

## Visualize Generated Image

### Subtask:
Display the rendered image using `matplotlib` to visually inspect the output of the NeRF model from the novel camera pose.


**Reasoning**:
The previous steps successfully rendered the novel view, but an OutOfMemoryError occurred. To address this, I will first retry rendering the novel view with the existing `chunk` and `chunk_mlp` values. If the error persists, I will try further reducing these values. Finally, I will visualize the rendered image as requested by the subtask.



In [ ]:
print("\n--- Attempting to render novel view with current settings ---")

# Set NeRF models to evaluation mode
nerf_coarse_model.eval()
if nerf_fine_model is not None:
    nerf_fine_model.eval()

# Call the render_image function with the novel_c2w
# Use smaller chunk sizes to prevent OOM errors
# H and W for render_image are from dummy_dataset
render_H, render_W = dummy_dataset['H'], dummy_dataset['W']

try:
    with torch.no_grad():
        novel_view_results = render_image(
            H=render_H, W=render_W, K=dummy_dataset['K'], c2w=novel_c2w,
            nerf_coarse=nerf_coarse_model,
            nerf_fine=nerf_fine_model,
            N_samples_coarse=N_samples_coarse,
            N_importance=N_importance,
            pos_encoder=pos_encoder,
            dir_encoder=dir_encoder,
            near=near_test, far=far_test,
            chunk=chunk, # This is N_rays_batch from training
            perturb=False, # No perturbation during rendering
            lindisp=False,
            white_bkgd=True,
            chunk_mlp=chunk_mlp # Chunk for MLP calls
        )
    print("Novel view rendered successfully.")

    # Display the rendered image
    import matplotlib.pyplot as plt

    # Get the fine rendered RGB map, detach from graph, move to CPU, and convert to NumPy
    rendered_image_np = novel_view_results['rgb_map_fine'].detach().cpu().numpy()

    plt.figure(figsize=(render_W/20, render_H/20)) # Adjust figure size based on image resolution
    plt.imshow(rendered_image_np, interpolation='nearest')
    plt.axis('off')
    plt.title('Rendered Novel View')
    plt.show()

except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print(f"CUDA Out of Memory Error during rendering: {e}")
        print("Consider reducing 'chunk' or 'chunk_mlp' even further, or reducing image resolution (H, W).")
    else:
        raise e



--- Attempting to render novel view with current settings ---
CUDA Out of Memory Error during rendering: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2433 has 14.74 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 55.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Consider reducing 'chunk' or 'chunk_mlp' even further, or reducing image resolution (H, W).


## Reduce NeRF Model and Data Parameters

### Subtask:
Reduce the image resolution, number of dummy images, ray sampling density, positional encoding frequencies, and NeRF MLP width and depth to decrease memory usage.


**Reasoning**:
I will set the new, reduced parameters for image dimensions, number of dummy images, ray sampling density, positional encoding frequencies, and NeRF MLP width and depth as specified in the instructions, and then print these new values.



In [ ]:
# 1. Set the image dimensions
H, W = 20, 20
print(f"New image dimensions: H={H}, W={W}")

# 2. Set the number of dummy images
N_images = 3
print(f"New number of dummy images: {N_images}")

# 3. Set the number of coarse samples per ray
N_samples_coarse = 16
print(f"New N_samples_coarse: {N_samples_coarse}")

# 4. Set the number of importance samples per ray
N_importance = 16
print(f"New N_importance: {N_importance}")

# 5. Set the positional encoding frequencies for positions
N_freqs_pos = 4
print(f"New N_freqs_pos: {N_freqs_pos}")

# 6. Set the positional encoding frequencies for view directions
N_freqs_dir = 2
print(f"New N_freqs_dir: {N_freqs_dir}")

# 7. Set the NeRF MLP width
W_mlp = 64
print(f"New W_mlp (MLP width): {W_mlp}")

# 8. Set the NeRF MLP depth
D_mlp = 4
print(f"New D_mlp (MLP depth): {D_mlp}")

# 9. Adjust the focal_length accordingly
focal_length = 1.2 * W
print(f"Adjusted focal_length: {focal_length}")


New image dimensions: H=20, W=20
New number of dummy images: 3
New N_samples_coarse: 16
New N_importance: 16
New N_freqs_pos: 4
New N_freqs_dir: 2
New W_mlp (MLP width): 64
New D_mlp (MLP depth): 4
Adjusted focal_length: 24.0


**Reasoning**:
Now that the parameters are reduced, I need to re-initialize the dummy dataset, positional encoders, and NeRF MLP models with these new, smaller values. This is crucial before attempting any training or rendering to ensure memory constraints are met.



In [ ]:
import torch
import numpy as np
import torch.nn.functional as F

# Ensure 'device' is defined for tensor placement
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Re-initializing data and models with new parameters on device: {device}")

# --- 1. Re-create the dummy_dataset with new H, W, N_images, and focal_length ---

# H, W, N_images, focal_length are now global from the previous cell

dummy_images = []
for _ in range(N_images):
    dummy_images.append(torch.rand(H, W, 3, device=device))

print(f"Generated {N_images} dummy ground truth images of size {H}x{W}.")

dummy_c2ws = []
radius = 4.0 # Keep radius consistent
for i in range(N_images):
    angle = 2 * np.pi * i / N_images
    cam_x = radius * np.sin(angle)
    cam_z = radius * np.cos(angle)
    cam_y = 0.5

    position = torch.tensor([cam_x, cam_y, cam_z], dtype=torch.float32, device=device)
    forward_vec = -F.normalize(position, dim=-1)
    up_vec = torch.tensor([0., 1., 0.], dtype=torch.float32, device=device)

    right_vec = F.normalize(torch.cross(up_vec, forward_vec, dim=-1), dim=-1)
    up_vec_corrected = F.normalize(torch.cross(forward_vec, right_vec, dim=-1), dim=-1)

    R = torch.stack([right_vec, up_vec_corrected, forward_vec], dim=1)

    c2w_matrix = torch.eye(4, device=device)
    c2w_matrix[:3, :3] = R
    c2w_matrix[:3, 3] = position

    dummy_c2ws.append(c2w_matrix)

print(f"Generated {N_images} dummy camera-to-world matrices.")

K = torch.tensor(
    [[focal_length, 0, W / 2],
     [0, focal_length, H / 2],
     [0, 0, 1]],
    dtype=torch.float32,
    device=device
)
print("Defined a single dummy camera intrinsic matrix K.")

dummy_dataset = {
    'images': dummy_images,
    'c2ws': dummy_c2ws,
    'K': K,
    'H': H,
    'W': W,
    'N_images': N_images
}
print("Re-created 'dummy_dataset' with new parameters.")

# --- 2. Re-instantiate PositionalEncoder and NeRFMLP models ---

# N_freqs_pos, N_freqs_dir, W_mlp, D_mlp are global from previous cell

pos_encoder = PositionalEncoder(N_freqs=N_freqs_pos).to(device)
dir_encoder = PositionalEncoder(N_freqs=N_freqs_dir).to(device)
print(f"Re-instantiated PositionalEncoders with N_freqs_pos={N_freqs_pos}, N_freqs_dir={N_freqs_dir}.")

# Calculate input dimensions for NeRFMLP with new positional encoders
dummy_coords_for_dim = torch.randn(1, 3).to(device)
pos_input_dim = pos_encoder(dummy_coords_for_dim).shape[-1]

dummy_viewdir_for_dim = torch.randn(1, 3).to(device) # Use 3D vector for directions
dir_input_dim = dir_encoder(dummy_viewdir_for_dim).shape[-1]

print(f"Calculated new pos_input_dim: {pos_input_dim}")
print(f"Calculated new dir_input_dim: {dir_input_dim}")

nerf_coarse_model = NeRFMLP(pos_input_dim, dir_input_dim, D=D_mlp, W=W_mlp, skips=[D_mlp//2]).to(device)
nerf_fine_model = NeRFMLP(pos_input_dim, dir_input_dim, D=D_mlp, W=W_mlp, skips=[D_mlp//2]).to(device)
print(f"Re-instantiated NeRFMLP coarse and fine models with D={D_mlp}, W={W_mlp}.")

print("All data and models re-initialized successfully.")



Re-initializing data and models with new parameters on device: cuda
Generated 3 dummy ground truth images of size 20x20.
Generated 3 dummy camera-to-world matrices.
Defined a single dummy camera intrinsic matrix K.
Re-created 'dummy_dataset' with new parameters.
Re-instantiated PositionalEncoders with N_freqs_pos=4, N_freqs_dir=2.
Calculated new pos_input_dim: 27
Calculated new dir_input_dim: 15
Re-instantiated NeRFMLP coarse and fine models with D=4, W=64.
All data and models re-initialized successfully.


## Perform a short training run

### Subtask:
Perform a short training run with the reduced parameters to get the NeRF models into a usable state.


**Reasoning**:
I will execute the training loop with the newly reduced parameters (`num_epochs = 1`, `N_rays_batch = 128`, `chunk = 32`, `chunk_mlp = 256`) to ensure the NeRF models are in a usable state, as specified in the instructions. This will involve moving models to the correct device, setting them to training mode, re-initializing the optimizer, and performing forward/backward passes with loss calculation for both coarse and fine models.



In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.notebook import tqdm
import random

# Ensure 'device' is defined for tensor placement
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Class and function definitions from previous cells to ensure self-contained block ---

class PositionalEncoder(nn.Module):
  def __init__(self, N_freqs, log_space=True):
    super().__init__()
    self.N_freqs = N_freqs
    self.log_space = log_space
    if log_space:
      self.freq_bands = 2.**torch.linspace(0., N_freqs - 1, N_freqs) * torch.pi
    else:
      self.freq_bands = torch.linspace(1., 2.**(N_freqs - 1), N_freqs) * torch.pi

  def forward(self, x):
    x_expanded = x.unsqueeze(-1)
    x_modulated = x_expanded * self.freq_bands.to(x.device)
    sin_encoded = torch.sin(x_modulated)
    cos_encoded = torch.cos(x_modulated)
    sin_encoded = sin_encoded.flatten(start_dim=-2)
    cos_encoded = cos_encoded.flatten(start_dim=-2)
    return torch.cat([x, sin_encoded, cos_encoded], dim=-1)

class NeRFMLP(nn.Module):
  def __init__(self, pos_input_dim, dir_input_dim, D=8, W=256, skips=[4]):
    super().__init__()
    self.D = D
    self.W = W
    self.skips = skips

    self.pts_linears = nn.ModuleList([nn.Linear(pos_input_dim, W)])

    for i in range(1, D):
        layer_in_dim = W
        if i in self.skips:
            layer_in_dim += pos_input_dim
        self.pts_linears.append(nn.Linear(layer_in_dim, W))

    self.alpha_linear = nn.Linear(W, 1)
    self.feature_linear = nn.Linear(W, W)

    self.views_linears = nn.ModuleList([nn.Linear(W + dir_input_dim, W // 2)])
    self.views_linears.extend([nn.Linear(W // 2, W // 2) for _ in range(2)])
    self.rgb_linear = nn.Linear(W // 2, 3)

  def forward(self, x, d):
    h = x

    for i, l in enumerate(self.pts_linears):
      if i in self.skips:
        h = torch.cat([x, h], -1)

      h = l(h)
      h = F.relu(h)

    alpha = self.alpha_linear(h)
    feature = self.feature_linear(h)

    h_views = torch.cat([feature, d], -1)
    for i, l in enumerate(self.views_linears):
      h_views = l(h_views)
      h_views = F.relu(h_views)
    rgb = self.rgb_linear(h_views)

    rgb = torch.sigmoid(rgb)
    alpha = F.relu(alpha)

    return rgb, alpha

def get_rays(H, W, focal, c2w):
  i, j = torch.meshgrid(
      torch.linspace(0, W - 1, W, device=c2w.device),
      torch.linspace(0, H - 1, H, device=c2w.device),
      indexing='xy'
  )
  dirs = torch.stack([
      (i - W * 0.5) / focal,
      -(j - H * 0.5) / focal,
      -torch.ones_like(i)
  ], -1)
  rays_d = torch.sum(dirs[..., None, :] * c2w[:3, :3], -1)
  rays_o = c2w[:3, 3].expand(rays_d.shape)
  rays_o = rays_o.reshape(-1, 3)
  rays_d = rays_d.reshape(-1, 3)
  return rays_o, rays_d

def sample_points_on_rays(rays_o, rays_d, near, far, N_samples, N_importance=0, perturb=True,
                          lindisp=False, coarse_model=None, positional_encoder_pos=None, positional_encoder_dir=None):
    N_rays = rays_o.shape[0]

    t_vals = torch.linspace(0., 1., N_samples, device=rays_o.device)
    if lindisp:
        t_vals = 1. / (1. / near * (1. - t_vals) + 1. / far * t_vals)
    else:
        t_vals = near * (1. - t_vals) + far * t_vals

    if perturb:
        mids = .5 * (t_vals[..., 1:] + t_vals[..., :-1])
        upper = torch.cat([mids, t_vals[..., -1:]], -1)
        lower = torch.cat([t_vals[..., :1], mids], -1)
        t_rand = torch.rand(N_samples, device=rays_o.device)
        t_vals = lower + (upper - lower) * t_rand

    pts_coarse = rays_o[..., None, :] + rays_d[..., None, :] * t_vals[..., :, None]
    z_vals_coarse = t_vals.expand(N_rays, N_samples)

    if N_importance > 0 and coarse_model is not None and positional_encoder_pos is not None and positional_encoder_dir is not None:
        with torch.no_grad():
            pts_flat = pts_coarse.reshape(-1, 3)
            encoded_pts_flat = positional_encoder_pos(pts_flat)
            rays_d_expanded = rays_d[..., None, :].expand(-1, N_samples, -1).reshape(-1, 3)
            encoded_rays_d_expanded = positional_encoder_dir(rays_d_expanded)
            _, raw_alpha = coarse_model(encoded_pts_flat, encoded_rays_d_expanded)

            dists = torch.cat([z_vals_coarse[..., 1:] - z_vals_coarse[..., :-1],
                               torch.tensor([1e10], device=rays_o.device).expand(z_vals_coarse[..., :1].shape)], -1)

            raw_alpha_reshaped = raw_alpha.reshape(N_rays, N_samples)

            alpha = 1. - torch.exp(-raw_alpha_reshaped * dists)
            weights = alpha * torch.cumprod(torch.cat([torch.ones((N_rays, 1), device=rays_o.device), 1.-alpha + 1e-10], -1), -1)[:, :-1]
            weights = weights + 1e-5

            pdf = weights / torch.sum(weights, -1, keepdim=True)
            cdf = torch.cumsum(pdf, -1)
            cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], -1)

        u = torch.rand(N_rays, N_importance, device=rays_o.device)
        u = u.contiguous()
        inds = torch.searchsorted(cdf, u, right=True)

        cdf_below_idx = torch.max(torch.zeros_like(inds - 1), inds - 1)
        cdf_above_idx = torch.min((cdf.shape[-1] - 1) * torch.ones_like(inds), inds)

        cdf_g_0 = torch.gather(cdf, -1, cdf_below_idx)
        cdf_g_1 = torch.gather(cdf, -1, cdf_above_idx)

        z_vals_below_idx = torch.clamp(inds - 1, min=0, max=N_samples - 1)
        z_vals_above_idx = torch.clamp(inds, min=0, max=N_samples - 1)

        z_vals_g_0 = torch.gather(z_vals_coarse, -1, z_vals_below_idx)
        z_vals_g_1 = torch.gather(z_vals_coarse, -1, z_vals_above_idx)

        denom = cdf_g_1 - cdf_g_0
        denom = torch.where(denom < 1e-5, torch.ones_like(denom), denom)
        t = (u - cdf_g_0) / denom
        z_vals_fine = z_vals_g_0 + t * (z_vals_g_1 - z_vals_g_0)

        z_vals_combined, _ = torch.sort(torch.cat([z_vals_coarse, z_vals_fine], -1), -1)

        pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals_combined[..., :, None]
        return pts, z_vals_combined
    else:
        return pts_coarse, z_vals_coarse

def render_rays(rgb, alpha, z_vals, rays_d):
    dists = z_vals[..., 1:] - z_vals[..., :-1]
    dists = torch.cat([dists, torch.tensor([1e10], device=dists.device).expand(dists[..., :1].shape)], -1)
    dists = dists * torch.norm(rays_d[..., None, :], dim=-1)

    alpha = 1. - torch.exp(-alpha.squeeze(-1) * dists)
    weights = alpha * torch.cumprod(torch.cat([torch.ones((alpha.shape[0], 1), device=alpha.device), 1.-alpha + 1e-10], -1), -1)[:, :-1]

    rgb_map = torch.sum(weights[..., None] * rgb, -2)

    depth_map = torch.sum(weights * z_vals, -1)
    depth_map = depth_map.unsqueeze(-1)

    acc_map = torch.sum(weights, -1)
    acc_map = acc_map.unsqueeze(-1)

    return rgb_map, depth_map, acc_map, weights.unsqueeze(-1)

def render_image(
    H,W,K,c2w,
    nerf_coarse,
    nerf_fine=None,
    N_samples_coarse=64,
    N_importance=0,
    pos_encoder=None,
    dir_encoder=None,
    near=0.,far=1.,
    chunk=1024*32, # This chunk refers to the maximum number of rays processed by render_image
    perturb=True,
    lindisp=False,
    white_bkgd=False,
    chunk_mlp=1024): # New parameter for chunking MLP calls

  focal=K[0,0]
  rays_o,rays_d=get_rays(H,W,focal,c2w.to(device))
  rays_o_flat=rays_o.reshape(-1,3)
  rays_d_flat=rays_d.reshape(-1,3)
  N_rays=rays_o_flat.shape[0]
  all_rgb_map_coarse=[]
  all_depth_map_coarse=[]
  all_acc_map_coarse=[]
  all_rgb_map_fine=[]
  all_depth_map_fine=[]
  all_acc_map_fine=[]

  # Process rays in smaller chunks, controlled by the 'chunk' parameter
  for i in range(0,N_rays,chunk):
    rays_o_chunk=rays_o_flat[i:i+chunk]
    rays_d_chunk=rays_d_flat[i:i+chunk]
    N_rays_chunk=rays_o_chunk.shape[0]

    pts_coarse,z_vals_coarse=sample_points_on_rays(
        rays_o_chunk,rays_d_chunk,near,far,N_samples_coarse,N_importance=0,perturb=perturb,lindisp=lindisp,
        coarse_model=nerf_coarse, positional_encoder_pos=pos_encoder, positional_encoder_dir=dir_encoder
    )

    pts_flat_coarse=pts_coarse.reshape(-1,3)
    N_points_coarse = pts_flat_coarse.shape[0] # N_rays_chunk * N_samples_coarse

    # Apply chunking for MLP calls within the current rays_chunk
    raw_rgb_coarse = []
    raw_alpha_coarse = []
    for j in range(0, N_points_coarse, chunk_mlp):
        pts_batch = pts_flat_coarse[j:j+chunk_mlp]

        # Calculate corresponding ray indices for this batch of points
        # Each point in pts_flat_coarse is (ray_idx * N_samples_coarse + sample_idx)
        # So, ray_idx = (current_point_global_idx) // N_samples_coarse
        # Correctly get the ray_indices_for_batch that correspond to the current pts_batch
        start_ray_idx = i // chunk # global ray index for the start of rays_o_chunk
        batch_global_indices = torch.arange(j, j + pts_batch.shape[0], device=device) # Indices within pts_flat_coarse for this chunk
        # The ray index for a point is its index in pts_flat_coarse // N_samples_coarse
        ray_indices_for_batch_local = batch_global_indices // N_samples_coarse
        dirs_batch = rays_d_chunk[ray_indices_for_batch_local]

        encoded_pts = pos_encoder(pts_batch) if pos_encoder else pts_batch
        encoded_dirs = dir_encoder(dirs_batch) if dir_encoder else dirs_batch

        rgb_chunk, alpha_chunk = nerf_coarse(encoded_pts, encoded_dirs)
        raw_rgb_coarse.append(rgb_chunk)
        raw_alpha_coarse.append(alpha_chunk)

    rgb_coarse = torch.cat(raw_rgb_coarse, 0).reshape(N_rays_chunk,N_samples_coarse,3)
    alpha_coarse = torch.cat(raw_alpha_coarse, 0).reshape(N_rays_chunk,N_samples_coarse,1)

    rgb_map_coarse,depth_map_coarse,acc_map_coarse,weights_coarse=render_rays(
        rgb_coarse,alpha_coarse,z_vals_coarse,rays_d_chunk
    )
    all_rgb_map_coarse.append(rgb_map_coarse)
    all_depth_map_coarse.append(depth_map_coarse)
    all_acc_map_coarse.append(acc_map_coarse)

    if N_importance > 0 and nerf_fine is not None:
      pts_combined,z_vals_combined=sample_points_on_rays(
          rays_o_chunk,rays_d_chunk,near,far,N_samples_coarse,
          N_importance=N_importance,perturb=perturb,lindisp=lindisp,
          coarse_model=nerf_coarse,positional_encoder_pos=pos_encoder,positional_encoder_dir=dir_encoder
      )
      N_total_samples=N_samples_coarse+N_importance
      pts_flat_combined=pts_combined.reshape(-1,3)
      N_points_fine = pts_flat_combined.shape[0] # N_rays_chunk * N_total_samples

      # Apply chunking for fine MLP calls
      raw_rgb_fine = []
      raw_alpha_fine = []
      for j in range(0, N_points_fine, chunk_mlp):
          pts_batch = pts_flat_combined[j:j+chunk_mlp]

          # Calculate corresponding ray indices for this batch of points
          start_ray_idx = i // chunk
          batch_global_indices = torch.arange(j, j + pts_batch.shape[0], device=device) # Indices within pts_flat_combined for this chunk
          ray_indices_for_batch_local = batch_global_indices // N_total_samples
          dirs_batch = rays_d_chunk[ray_indices_for_batch_local]

          encoded_pts = pos_encoder(pts_batch) if pos_encoder else pts_batch
          encoded_dirs = dir_encoder(dirs_batch) if dir_encoder else dirs_batch

          rgb_chunk, alpha_chunk = nerf_fine(encoded_pts, encoded_dirs)
          raw_rgb_fine.append(rgb_chunk)
          raw_alpha_fine.append(alpha_chunk)

      rgb_fine=torch.cat(raw_rgb_fine, 0).reshape(N_rays_chunk,N_total_samples,3)
      alpha_fine=torch.cat(raw_alpha_fine, 0).reshape(N_rays_chunk,N_total_samples,1)

      rgb_map_fine,depth_map_fine,acc_map_fine,_=render_rays(
          rgb_fine,alpha_fine,z_vals_combined,rays_d_chunk
      )
      all_rgb_map_fine.append(rgb_map_fine)
      all_depth_map_fine.append(depth_map_fine)
      all_acc_map_fine.append(acc_map_fine)

  rgb_map_coarse=torch.cat(all_rgb_map_coarse,0).reshape(H,W,3)
  depth_map_coarse=torch.cat(all_depth_map_coarse,0).reshape(H,W,1)
  acc_map_coarse=torch.cat(all_acc_map_coarse,0).reshape(H,W,1)

  results={
      'rgb_map_coarse':rgb_map_coarse,
      "depth_map_coarse":depth_map_coarse,
      "acc_map_coarse":acc_map_coarse
  }

  if N_importance > 0 and nerf_fine is not None:
    rgb_map_fine=torch.cat(all_rgb_map_fine,0).reshape(H,W,3)
    depth_map_fine=torch.cat(all_depth_map_fine,0).reshape(H,W,1)
    acc_map_fine=torch.cat(all_acc_map_fine,0).reshape(H,W,1)
    results['rgb_map_fine']=rgb_map_fine
    results['depth_map_fine']=depth_map_fine
    results['acc_map_fine']=acc_map_fine
    if white_bkgd:
      results['rgb_map_fine']=results['rgb_map_fine']+(1.-results['acc_map_fine'])
      results['rgb_map_fine']=torch.clamp(results['rgb_map_fine'],0.,1.)
  return results

# --- Training Loop --- START

# 1. Set training parameters
num_epochs = 1
N_rays_batch = 128 # Reduced from 256
chunk = 32 # For render_image's outer loop (rays)
chunk_mlp = 256 # For render_image's inner MLP calls (points)
learning_rate = 5e-4

print(f"Starting NeRF training for {num_epochs} epoch(s) with N_rays_batch={N_rays_batch}, chunk={chunk}, chunk_mlp={chunk_mlp}...")

# 2. Ensure models are on the correct device and in training mode
nerf_coarse_model.to(device).train()
if nerf_fine_model is not None:
  nerf_fine_model.to(device).train()

# 3. Re-initialize the optimizer with the parameters of the re-instantiated coarse and fine models.
all_model_params = list(nerf_coarse_model.parameters())
if nerf_fine_model is not None:
    all_model_params.extend(list(nerf_fine_model.parameters()))
optimizer = optim.Adam(all_model_params, lr=learning_rate)

criterion = torch.nn.MSELoss() # Ensure criterion is defined

# 4. Execute the training loop
for epoch in tqdm(range(num_epochs), desc="Epoch Loop"):
    img_idx = random.randint(0, dummy_dataset['N_images'] - 1)
    target_img = dummy_dataset['images'][img_idx]
    target_c2w = dummy_dataset['c2ws'][img_idx]
    target_K = dummy_dataset['K']
    H, W = dummy_dataset['H'], dummy_dataset['W'] # Use the globally defined H, W
    focal = target_K[0, 0]

    rays_o_all, rays_d_all = get_rays(H, W, focal, target_c2w)
    target_rgb_flat = target_img.reshape(-1, 3)
    total_rays_in_image = rays_o_all.shape[0]

    # Create a shuffled list of indices for all rays in the image
    rand_idx_all = torch.randperm(total_rays_in_image, device=device)

    for i in tqdm(range(0, total_rays_in_image, N_rays_batch), desc=f"Batch Loop (Epoch {epoch+1})"):
        current_batch_size_training = min(N_rays_batch, total_rays_in_image - i)
        if current_batch_size_training == 0:
            continue

        # Select a batch of rays and corresponding target RGB values
        batch_indices = rand_idx_all[i : i + current_batch_size_training]
        batch_rays_o = rays_o_all[batch_indices]
        batch_rays_d = rays_d_all[batch_indices]
        batch_target_rgb = target_rgb_flat[batch_indices]

        # Render image for the current batch of rays
        rendered_outputs = render_image(
            H=H, W=W, K=target_K, c2w=target_c2w,
            nerf_coarse=nerf_coarse_model,
            nerf_fine=nerf_fine_model,
            N_samples_coarse=N_samples_coarse,
            N_importance=N_importance,
            pos_encoder=pos_encoder,
            dir_encoder=dir_encoder,
            near=near_test, far=far_test,
            chunk=chunk, # Use the chunk value for render_image's outer loop
            perturb=True,
            lindisp=False,
            white_bkgd=True,
            chunk_mlp=chunk_mlp # Pass the new MLP chunk size
        )

        loss = 0.0
        # Compute loss for fine network if available
        if nerf_fine_model is not None:
            # The render_image function now returns H,W,3 or H,W,1.
            # Need to reshape the fine RGB output from render_image to match batch_target_rgb for loss calculation.
            # The output rgb_map_fine in `rendered_outputs` is (chunk, 3) because it's already flattened within render_image's internal chunking.
            rgb_map_fine = rendered_outputs['rgb_map_fine'].reshape(-1, 3) # ensure it's (N_rays_in_chunk, 3)
            loss_fine = criterion(rgb_map_fine, batch_target_rgb)
            loss += loss_fine

        # Compute loss for coarse network
        rgb_map_coarse = rendered_outputs['rgb_map_coarse'].reshape(-1, 3)
        loss_coarse = criterion(rgb_map_coarse, batch_target_rgb)
        loss += loss_coarse

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update tqdm description with current loss
        tqdm.write(f"Epoch {epoch+1}/{num_epochs}, Batch {i//N_rays_batch+1}/{ (total_rays_in_image + N_rays_batch - 1) // N_rays_batch}, Total Loss: {loss.item():.4f}")

print("\n--- Short NeRF Training Complete with reduced parameters ---")

# --- Training Loop --- END

Starting NeRF training for 1 epoch(s) with N_rays_batch=128, chunk=32, chunk_mlp=256...


Epoch Loop:   0%|          | 0/1 [00:00<?, ?it/s]

Batch Loop (Epoch 1):   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2433 has 14.74 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 54.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)